CATEGORY SEMEMANTIC DECOMPOSITION

In [34]:

import torch
import torch.nn as nn
import torch.nn.functional as F

class CategorySemanticDecomposition(nn.Module):

    def __init__(self, input_dim=10, hidden_dim=64):
        super().__init__()

        assert hidden_dim % 2 == 0

        half_dim = hidden_dim // 2

        # Equation 1: shared feature projection
        self.input_projection = nn.Linear(input_dim, hidden_dim)

        # Equation 2: separate semantic projections
        self.normal_projection = nn.Linear(half_dim, half_dim)
        self.abnormal_projection = nn.Linear(half_dim, half_dim)

        self.activation = nn.LeakyReLU()

    def forward(self, x):

        # Equation 1
        h = self.activation(self.input_projection(x))

        # Divide into two equal feature parts
        first_half, second_half = torch.chunk(h, 2, dim=1)

        # Equation 2
        normal = self.normal_projection(first_half)
        abnormal = self.abnormal_projection(second_half)

        return normal, abnormal

In [4]:
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
PROCESSED_DIR = PROJECT_DIR / "notebooks" / "data" / "processed"

print("Current directory:", Path.cwd())
print("Processed directory:", PROCESSED_DIR)
print("\nFiles found:")

for file in PROCESSED_DIR.iterdir():
    print(file.name)

Current directory: c:\Users\User\OneDrive\Desktop\Faud detection with limited resources\Fraude_detection_with_limited_resources\notebooks
Processed directory: c:\Users\User\OneDrive\Desktop\Faud detection with limited resources\Fraude_detection_with_limited_resources\notebooks\data\processed

Files found:
experimental_splits.pt
tfinance_processed.bin
tfinance_pyg_export.pt


In [5]:
import torch
from torch_geometric.data import Data

export = torch.load(
    PROCESSED_DIR / "tfinance_pyg_export.pt",
    map_location="cpu",
    weights_only=True
)

data = Data(
    x=export["x"],
    y=export["y"],
    edge_index=export["edge_index"]
)

print(data)

Data(x=[39357, 10], edge_index=[2, 42445086], y=[39357])


In [6]:
splits = torch.load(
    PROCESSED_DIR / "experimental_splits.pt",
    map_location="cpu",
    weights_only=True
)

train_idx = splits["train_idx"]
val_idx = splits["val_idx"]
test_idx = splits["test_idx"]

labelled_1 = splits["labelled_1"]
unlabelled_1 = splits["unlabelled_1"]

labelled_5 = splits["labelled_5"]
unlabelled_5 = splits["unlabelled_5"]

seed = splits["seed"]

print("Graph and experimental splits loaded!")

Graph and experimental splits loaded!


In [35]:
csd = CategorySemanticDecomposition(
    input_dim=10,
    hidden_dim=64
).to(device)

normal_new, abnormal_new = csd(data.x.to(device))

print("Input:", data.x.shape)
print("Normal representation:", normal_new.shape)
print("Abnormal representation:", abnormal_new.shape)

print("CSD parameters:", sum(
    p.numel() for p in csd.parameters()
))

print("Contains NaN:", torch.isnan(normal_new).any().item()
      or torch.isnan(abnormal_new).any().item())

print("Contains Inf:", torch.isinf(normal_new).any().item()
      or torch.isinf(abnormal_new).any().item())

Input: torch.Size([39357, 10])
Normal representation: torch.Size([39357, 32])
Abnormal representation: torch.Size([39357, 32])
CSD parameters: 2816
Contains NaN: False
Contains Inf: False


DENOISING ATTENTION


In [36]:
from torch_geometric.utils import softmax

class DenoisingAttention(nn.Module):

    def __init__(self, hidden_dim=32, attention_dim=32):
        super().__init__()

        # W and b in Equation 4
        self.attention_projection = nn.Linear(
            hidden_dim, attention_dim
        )

        # W_Att^T in Equation 4: one score per neighbour
        self.attention_score = nn.Linear(
            attention_dim, 1, bias=False
        )

    def forward(self, normal, abnormal, edge_index):

        src, dst = edge_index

        # Combine the two subspace features of each neighbour
        combined = normal[src] + abnormal[src]

        # Calculate attention scores
        hidden = torch.tanh(
            self.attention_projection(combined)
        )

        scores = self.attention_score(hidden)

        # Softmax across neighbours of the same centre node
        alpha = softmax(
            scores,
            index=dst,
            num_nodes=normal.size(0)
        )

        # Equation 4: denoised neighbour messages
        messages = (
            alpha * normal[src]
            + (1 - alpha) * abnormal[src]
        )

        return messages, alpha

In [37]:
# Use the corrected CSD representations
edge_index_test = data.edge_index[:, :100_000].contiguous().to(device)

# Initialize corrected attention
attention_model = DenoisingAttention(
    hidden_dim=32,
    attention_dim=32
).to(device)

# Forward pass
messages, alpha_new = attention_model(
    normal_new,
    abnormal_new,
    edge_index_test
)

print("Normal features:", normal_new.shape)
print("Abnormal features:", abnormal_new.shape)
print("Test edges:", edge_index_test.shape)
print("Attention weights:", alpha_new.shape)
print("Denoised messages:", messages.shape)

print("Minimum attention:", alpha_new.min().item())
print("Maximum attention:", alpha_new.max().item())

print("Contains NaN:", torch.isnan(messages).any().item())
print("Contains Inf:", torch.isinf(messages).any().item())

Normal features: torch.Size([39357, 32])
Abnormal features: torch.Size([39357, 32])
Test edges: torch.Size([2, 100000])
Attention weights: torch.Size([100000, 1])
Denoised messages: torch.Size([100000, 32])
Minimum attention: 0.02321750856935978
Maximum attention: 1.0
Contains NaN: False
Contains Inf: False


AGGREGATION AND UPDATE


In [38]:
class AggregationUpdate(nn.Module):
    def __init__(self, hidden_dim=32):
        super().__init__()

        self.update_projection = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, center_features, messages, edge_index):

        num_nodes = center_features.size(0)

        src, dst = edge_index

        # Step 1: Sum neighbour messages for each centre node
        neighbour_sum = torch.zeros_like(center_features)

        neighbour_sum.index_add_(
            0,
            dst,
            messages
        )

        # Step 2: Calculate the degree of each centre node
        degree = torch.bincount(
            dst,
            minlength=num_nodes
        ).to(center_features.dtype)

        # Step 3: Normalize by square root of degree
        degree_norm = degree.clamp(min=1).sqrt().unsqueeze(1)

        # Step 4: Add centre features and aggregated messages
        combined = center_features + neighbour_sum / degree_norm

        # Step 5: Fully connected layer
        updated_features = self.update_projection(combined)

        return updated_features

In [40]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cpu


In [41]:
# Combine the two CSD representations for this initial test
center_features = normal_new + abnormal_new

# Initialize aggregation layer
aggregation_model = AggregationUpdate(hidden_dim=32).to(device)

# Run aggregation
updated_features = aggregation_model(
    center_features,
    messages,
    edge_index_test
)

print("Centre features:", center_features.shape)
print("Neighbour messages:", messages.shape)
print("Updated features:", updated_features.shape)

print("NaN values:", torch.isnan(updated_features).any().item())
print("Infinite values:", torch.isinf(updated_features).any().item())
print("Parameters:", sum(p.numel() for p in aggregation_model.parameters()))

Centre features: torch.Size([39357, 32])
Neighbour messages: torch.Size([100000, 32])
Updated features: torch.Size([39357, 32])
NaN values: False
Infinite values: False
Parameters: 1056


In [42]:
class CGNNClassifier(nn.Module):
    def __init__(self, input_dim=32, hidden_dim=64, num_classes=2):
        super().__init__()

        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.classifier(x)

In [43]:
# Initialize classifier
classifier = CGNNClassifier(
    input_dim=32,
    hidden_dim=64,
    num_classes=2
).to(device)

# Generate predictions from updated node features
logits = classifier(updated_features)

# Convert logits into predicted classes
predictions = torch.argmax(logits, dim=1)

print("Updated features:", updated_features.shape)
print("Logits:", logits.shape)
print("Predictions:", predictions.shape)

print("Predicted normal nodes:", (predictions == 0).sum().item())
print("Predicted fraud nodes:", (predictions == 1).sum().item())

print("NaN values:", torch.isnan(logits).any().item())
print("Infinite values:", torch.isinf(logits).any().item())

print("Classifier parameters:",
      sum(p.numel() for p in classifier.parameters()))

Updated features: torch.Size([39357, 32])
Logits: torch.Size([39357, 2])
Predictions: torch.Size([39357])
Predicted normal nodes: 39057
Predicted fraud nodes: 300
NaN values: False
Infinite values: False
Classifier parameters: 2242


In [14]:
class CGNNModel(nn.Module):
    def __init__(self, input_dim=10, hidden_dim=64,
                 num_classes=2):
        super().__init__()

        # 1. Category Semantic Decomposition
        self.csd = CategorySemanticDecomposition(
            input_dim=input_dim,
            hidden_dim=hidden_dim
        )

        # 2. Denoising Attention
        self.denoising_attention = DenoisingAttention(
            hidden_dim=hidden_dim // 2
        )

        # 3. Aggregation and Update
        self.aggregation = AggregationUpdate(
            hidden_dim=hidden_dim // 2
        )

        # 4. MLP Classifier
        self.classifier = CGNNClassifier(
            input_dim=hidden_dim // 2,
            hidden_dim=64,
            num_classes=num_classes
        )

    def forward(self, x, edge_index):

        # Step 1: Decompose node features
        normal, abnormal = self.csd(x)

        # Step 2: Denoise representations
        denoised, attention = self.denoising_attention(
            normal, abnormal
        )

        # Step 3: Aggregate neighbour information
        updated = self.aggregation(
            denoised, edge_index
        )

        # Step 4: Predict classes
        logits = self.classifier(updated)

        return logits, normal, abnormal, denoised, attention

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cpu


In [17]:
# Create the complete CGNN model
model = CGNNModel(
    input_dim=10,
    hidden_dim=64,
    num_classes=2
).to(device)

# Move graph data to the device
x = data.x.to(device)

# Use the first 100,000 edges for this initial test
edge_index_test = data.edge_index[:, :100_000].to(device)

# Forward pass
model.eval()

with torch.no_grad():
    logits, normal, abnormal, denoised, attention = model(
        x, edge_index_test
    )

# Display output shapes
print("Normal representation:", normal.shape)
print("Abnormal representation:", abnormal.shape)
print("Denoised representation:", denoised.shape)
print("Attention weights:", attention.shape)
print("Logits:", logits.shape)

print("\nTotal model parameters:",
      sum(p.numel() for p in model.parameters()))

print("Contains NaN:", torch.isnan(logits).any().item())
print("Contains Inf:", torch.isinf(logits).any().item())

Normal representation: torch.Size([39357, 32])
Abnormal representation: torch.Size([39357, 32])
Denoised representation: torch.Size([39357, 32])
Attention weights: torch.Size([39357, 1])
Logits: torch.Size([39357, 2])

Total model parameters: 7236
Contains NaN: False
Contains Inf: False


In [44]:
# Get labels of the labelled training nodes
labels = data.y[labelled_1]

# Count samples in each class
class_counts = torch.bincount(labels, minlength=2).float()

# Calculate class weights
class_weights = len(labels) / (2 * class_counts)

class_weights = class_weights.to(device)

print("Class counts:", class_counts)
print("Class weights:", class_weights)

Class counts: tensor([21.,  1.])
Class weights: tensor([ 0.5238, 11.0000])


In [45]:
# Initialize weighted cross-entropy loss
criterion = nn.CrossEntropyLoss(weight=class_weights)

# Select logits of labelled nodes only
labelled_logits = logits[labelled_1].to(device)

# Get their actual labels
labelled_targets = data.y[labelled_1].to(device)

# Calculate loss
ce_loss = criterion(labelled_logits, labelled_targets)

print("Cross-Entropy Loss:", ce_loss.item())

Cross-Entropy Loss: 1.023166537284851


In [46]:
def csd_contrastive_loss(
    normal_features,
    abnormal_features,
    labels,
    labelled_indices,
    tau=0.8
):
    # Get labels of selected labelled nodes
    y = labels[labelled_indices]

    # Select the relevant subspace for each category
    features = torch.where(
        (y == 0).unsqueeze(1),
        normal_features[labelled_indices],
        abnormal_features[labelled_indices]
    )

    # Normalize feature vectors
    features = F.normalize(features, p=2, dim=1)

    # Calculate cosine similarity
    similarity = torch.mm(features, features.T) / tau

    # Identify positive and negative pairs
    same_class = y.unsqueeze(0) == y.unsqueeze(1)

    self_mask = torch.eye(
        len(y),
        dtype=torch.bool,
        device=y.device
    )

    positive_mask = same_class & ~self_mask
    negative_mask = ~same_class

    # Keep anchors that have both positive and negative pairs
    valid = positive_mask.any(dim=1) & negative_mask.any(dim=1)

    if not valid.any():
        return features.sum() * 0.0

    losses = []

    for k in torch.where(valid)[0]:

        positive_scores = similarity[k][positive_mask[k]]
        negative_scores = similarity[k][negative_mask[k]]

        denominator = torch.logsumexp(negative_scores, dim=0)

        pair_losses = -(positive_scores - denominator)

        losses.append(pair_losses.mean())

    return torch.stack(losses).mean()

In [47]:
csd_loss = csd_contrastive_loss(
    normal_features=normal_new,
    abnormal_features=abnormal_new,
    labels=data.y,
    labelled_indices=labelled_5,
    tau=0.8
)

print("CSD Contrastive Loss:", csd_loss.item())
print("NaN values:", torch.isnan(csd_loss).item())
print("Infinite values:", torch.isinf(csd_loss).item())

CSD Contrastive Loss: 1.100529670715332
NaN values: False
Infinite values: False


In [24]:
# Create fresh copies so we do not affect the model's existing gradients
normal_test = normal.detach().clone().requires_grad_(True)
abnormal_test = abnormal.detach().clone().requires_grad_(True)

# Calculate CSD contrastive loss
csd_loss_test = csd_contrastive_loss(
    normal_features=normal_test,
    abnormal_features=abnormal_test,
    labels=data.y.to(normal.device),
    labelled_indices=labelled_5.to(normal.device),
    tau=0.8
)

# Backpropagation
csd_loss_test.backward()

# Check gradients
print("CSD Loss:", csd_loss_test.item())

print("Normal gradients exist:", normal_test.grad is not None)
print("Abnormal gradients exist:", abnormal_test.grad is not None)

print("Normal gradient norm:", normal_test.grad.norm().item())
print("Abnormal gradient norm:", abnormal_test.grad.norm().item())

print("Normal gradients finite:", torch.isfinite(normal_test.grad).all().item())
print("Abnormal gradients finite:", torch.isfinite(abnormal_test.grad).all().item())

CSD Loss: 1.1165077686309814
Normal gradients exist: True
Abnormal gradients exist: True
Normal gradient norm: 0.002117051277309656
Abnormal gradient norm: 0.02413303218781948
Normal gradients finite: True
Abnormal gradients finite: True


SUBSPACE FEATURE PERTURBATION

In [48]:
class SubspaceFeaturePerturbation(nn.Module):

    def forward(self, normal, abnormal, alpha, edge_index):

        src, dst = edge_index

        # Get the normal and abnormal features of each neighbour
        normal_edges = normal[src]
        abnormal_edges = abnormal[src]

        # Randomly permute feature dimensions within each subspace
        normal_indices = torch.rand(
            normal_edges.shape, device=normal.device
        ).argsort(dim=1)

        abnormal_indices = torch.rand(
            abnormal_edges.shape, device=abnormal.device
        ).argsort(dim=1)

        normal_fa = torch.gather(
            normal_edges, 1, normal_indices
        )

        abnormal_fa = torch.gather(
            abnormal_edges, 1, abnormal_indices
        )

        # Reshape edge attention weights for feature-wise combination
        alpha = alpha.view(-1, 1)

        # Equation 9: create two augmented message views
        normal_perturbed = (
            alpha * normal_fa
            + (1 - alpha) * abnormal_edges
        )

        abnormal_perturbed = (
            alpha * normal_edges
            + (1 - alpha) * abnormal_fa
        )

        return normal_perturbed, abnormal_perturbed

In [27]:
# Extract alpha from the denoising attention output
alpha = attention

print("Alpha shape:", alpha.shape)

Alpha shape: torch.Size([39357, 1])


In [49]:
sfp = SubspaceFeaturePerturbation().to(device)

normal_perturbed, abnormal_perturbed = sfp(
    normal=normal_new,
    abnormal=abnormal_new,
    alpha=alpha_new,
    edge_index=edge_index_test
)

print("Normal perturbed messages:", normal_perturbed.shape)
print("Abnormal perturbed messages:", abnormal_perturbed.shape)

print(
    "Normal NaN:",
    torch.isnan(normal_perturbed).any().item()
)

print(
    "Abnormal NaN:",
    torch.isnan(abnormal_perturbed).any().item()
)

print(
    "Normal Inf:",
    torch.isinf(normal_perturbed).any().item()
)

print(
    "Abnormal Inf:",
    torch.isinf(abnormal_perturbed).any().item()
)

Normal perturbed messages: torch.Size([100000, 32])
Abnormal perturbed messages: torch.Size([100000, 32])
Normal NaN: False
Abnormal NaN: False
Normal Inf: False
Abnormal Inf: False


CONSISTENCY LOSS

In [50]:
def consistency_loss(
    y_hat,
    y_hat_0,
    y_hat_1,
    unlabelled_indices
):

    # Select predictions for unlabelled nodes
    original = y_hat[unlabelled_indices]
    normal_view = y_hat_0[unlabelled_indices]
    abnormal_view = y_hat_1[unlabelled_indices]

    # Calculate cosine similarities
    sim_normal = F.cosine_similarity(
        original, normal_view, dim=1
    )

    sim_abnormal = F.cosine_similarity(
        original, abnormal_view, dim=1
    )

    # Avoid log(0)
    eps = 1e-8
    sim_abnormal = sim_abnormal.clamp(
        min=eps, max=1.0
    )

    # Equation 10
    loss = -(sim_normal * torch.log(sim_abnormal)).mean()

    return loss

In [51]:
def aggregate_messages(messages, edge_index, num_nodes):

    src, dst = edge_index

    aggregated = torch.zeros(
        (num_nodes, messages.size(1)),
        device=messages.device
    )

    aggregated.index_add_(0, dst, messages)

    degree = torch.bincount(
        dst, minlength=num_nodes
    ).to(messages.dtype)

    return aggregated / degree.clamp(min=1).sqrt().unsqueeze(1)


# Original prediction
original_logits = classifier(updated_features)

# Aggregate the two perturbed views
normal_view_features = aggregate_messages(
    normal_perturbed, edge_index_test, data.num_nodes
)

abnormal_view_features = aggregate_messages(
    abnormal_perturbed, edge_index_test, data.num_nodes
)

# Apply the classifier to both views
normal_view_logits = classifier(
    center_features + normal_view_features
)

abnormal_view_logits = classifier(
    center_features + abnormal_view_features
)

print("Original logits:", original_logits.shape)
print("Normal view logits:", normal_view_logits.shape)
print("Abnormal view logits:", abnormal_view_logits.shape)

Original logits: torch.Size([39357, 2])
Normal view logits: torch.Size([39357, 2])
Abnormal view logits: torch.Size([39357, 2])


In [52]:
cl_loss = consistency_loss(
    y_hat=original_logits,
    y_hat_0=normal_view_logits,
    y_hat_1=abnormal_view_logits,
    unlabelled_indices=unlabelled_5
)

print("Consistency Loss:", cl_loss.item())
print("NaN values:", torch.isnan(cl_loss).item())
print("Infinite values:", torch.isinf(cl_loss).item())

Consistency Loss: 0.24644792079925537
NaN values: False
Infinite values: False


In [53]:
def cgnn_total_loss(
    ce_loss,
    csd_loss,
    cl_loss,
    lambda_csd=1.0,
    beta_cl=1.0
):
    total_loss = (
        ce_loss
        + lambda_csd * csd_loss
        + beta_cl * cl_loss
    )

    return total_loss

In [54]:
total_loss = cgnn_total_loss(
    ce_loss,
    csd_loss,
    cl_loss,
    lambda_csd=1.0,
    beta_cl=1.0
)

print("Cross-Entropy Loss:", ce_loss.item())
print("CSD Loss:", csd_loss.item())
print("Consistency Loss:", cl_loss.item())
print("Total CGNN Loss:", total_loss.item())

print("NaN:", torch.isnan(total_loss).item())
print("Infinite:", torch.isinf(total_loss).item())

Cross-Entropy Loss: 1.023166537284851
CSD Loss: 1.100529670715332
Consistency Loss: 0.24644792079925537
Total CGNN Loss: 2.3701443672180176
NaN: False
Infinite: False


In [30]:
def consistency_loss(y_hat, y_hat_0, y_hat_1, unlabelled_indices):

    # Select predictions for unlabelled nodes
    original = y_hat[unlabelled_indices]
    normal_view = y_hat_0[unlabelled_indices]
    abnormal_view = y_hat_1[unlabelled_indices]

    # Cosine similarities between the three prediction views
    sim_normal = F.cosine_similarity(
        original, normal_view, dim=1
    )

    sim_abnormal = F.cosine_similarity(
        original, abnormal_view, dim=1
    )

    # Numerical stability for logarithm
    eps = 1e-8
    sim_abnormal = sim_abnormal.clamp(min=eps, max=1.0)

    # Equation 10
    loss = -(sim_normal * torch.log(sim_abnormal)).mean()

    return loss

In [55]:
import psutil

memory = psutil.virtual_memory()

print("Graph nodes:", data.num_nodes)
print("Graph edges:", data.num_edges)

print(f"Total RAM: {memory.total / (1024**3):.2f} GB")
print(f"Available RAM: {memory.available / (1024**3):.2f} GB")
print(f"Used RAM: {memory.used / (1024**3):.2f} GB")

Graph nodes: 39357
Graph edges: 42445086
Total RAM: 15.75 GB
Available RAM: 1.06 GB
Used RAM: 14.70 GB


In [56]:
import gc
import torch

# Remove temporary test variables
del edge_index_test
del messages
del alpha_new
del normal_perturbed
del abnormal_perturbed

del normal_view_features
del abnormal_view_features
del normal_view_logits
del abnormal_view_logits

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Temporary variables cleared.")

Temporary variables cleared.


In [57]:
import psutil

memory = psutil.virtual_memory()

print(f"Available RAM: {memory.available / (1024**3):.2f} GB")
print(f"Used RAM: {memory.used / (1024**3):.2f} GB")

Available RAM: 1.02 GB
Used RAM: 14.73 GB


In [1]:
from pathlib import Path
import torch
import gc
import psutil

from torch_geometric.data import Data

PROJECT_DIR = Path.cwd().parent
PROCESSED_DIR = PROJECT_DIR / "notebooks" / "data" / "processed"

device = torch.device("cpu")

export = torch.load(
    PROCESSED_DIR / "tfinance_pyg_export.pt",
    map_location="cpu",
    weights_only=True
)

data = Data(
    x=export["x"],
    y=export["y"],
    edge_index=export["edge_index"]
)

splits = torch.load(
    PROCESSED_DIR / "experimental_splits.pt",
    map_location="cpu",
    weights_only=True
)

print(data)
print(f"Available RAM: {psutil.virtual_memory().available / (1024**3):.2f} GB")

Data(x=[39357, 10], edge_index=[2, 42445086], y=[39357])
Available RAM: 1.39 GB


In [2]:
splits = torch.load(
    PROCESSED_DIR / "experimental_splits.pt",
    map_location="cpu",
    weights_only=True
)

train_idx = splits["train_idx"]
val_idx = splits["val_idx"]
test_idx = splits["test_idx"]

labelled_1 = splits["labelled_1"]
unlabelled_1 = splits["unlabelled_1"]

labelled_5 = splits["labelled_5"]
unlabelled_5 = splits["unlabelled_5"]

seed = splits["seed"]

print("Training nodes:", len(train_idx))
print("Validation nodes:", len(val_idx))
print("Testing nodes:", len(test_idx))

print("\n1-fraud setting:")
print("Labelled:", len(labelled_1))
print("Unlabelled:", len(unlabelled_1))

print("\n5-fraud setting:")
print("Labelled:", len(labelled_5))
print("Unlabelled:", len(unlabelled_5))

Training nodes: 3935
Validation nodes: 11807
Testing nodes: 23615

1-fraud setting:
Labelled: 22
Unlabelled: 3913

5-fraud setting:
Labelled: 109
Unlabelled: 3826


In [3]:
import importlib.util

print("pyg-lib installed:",
      importlib.util.find_spec("pyg_lib") is not None)

print("torch-sparse installed:",
      importlib.util.find_spec("torch_sparse") is not None)

print("\nSplit sizes:")
print("Train:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))

print("\nLabelled 1:", len(labelled_1))
print("Labelled 5:", len(labelled_5))

pyg-lib installed: False
torch-sparse installed: False

Split sizes:
Train: 3935
Validation: 11807
Test: 23615

Labelled 1: 22
Labelled 5: 109


In [8]:
import torch
import torch_geometric
import torch_sparse
import torch_scatter

print("PyTorch:", torch.__version__)
print("PyG:", torch_geometric.__version__)
print("torch_sparse imported:", torch_sparse.__version__)
print("torch_scatter imported:", torch_scatter.__version__)

from torch_geometric.typing import WITH_TORCH_SPARSE
print("PyG detects torch_sparse:", WITH_TORCH_SPARSE)

PyTorch: 2.2.1+cpu
PyG: 2.8.1
torch_sparse imported: 0.6.18+pt22cpu
torch_scatter imported: 2.1.2+pt22cpu
PyG detects torch_sparse: False


In [9]:
import torch

def manual_neighbor_sampling(
    edge_index,
    seed_nodes,
    num_neighbors=10,
    num_nodes=None
):
    """
    Sample incoming neighbours for a batch of seed nodes.
    Uses pure PyTorch.
    """

    src, dst = edge_index

    if num_nodes is None:
        num_nodes = int(edge_index.max()) + 1

    sampled_edges = []
    sampled_nodes = set(seed_nodes.tolist())

    for node in seed_nodes.tolist():

        # Find incoming neighbours of this node
        neighbors = src[dst == node]

        # Randomly select up to num_neighbors
        if len(neighbors) > num_neighbors:
            selected = torch.randperm(len(neighbors))[:num_neighbors]
            neighbors = neighbors[selected]

        for neighbor in neighbors.tolist():
            sampled_edges.append((neighbor, node))
            sampled_nodes.add(neighbor)

    if sampled_edges:
        sampled_edge_index = torch.tensor(
            sampled_edges,
            dtype=torch.long
        ).t().contiguous()
    else:
        sampled_edge_index = torch.empty((2, 0), dtype=torch.long)

    sampled_nodes = torch.tensor(
        sorted(sampled_nodes),
        dtype=torch.long
    )

    return sampled_nodes, sampled_edge_index

In [10]:
seed_nodes = train_idx[:16]

sampled_nodes, sampled_edge_index = manual_neighbor_sampling(
    edge_index=data.edge_index,
    seed_nodes=seed_nodes,
    num_neighbors=10
)

print("Seed nodes:", len(seed_nodes))
print("Sampled nodes:", len(sampled_nodes))
print("Sampled edges:", sampled_edge_index.shape[1])
print("Sampled edge shape:", sampled_edge_index.shape)

Seed nodes: 16
Sampled nodes: 176
Sampled edges: 160
Sampled edge shape: torch.Size([2, 160])


In [11]:
# Build an incoming-neighbour index
src, dst = data.edge_index

# Sort edges by destination node
sort_idx = torch.argsort(dst)

sorted_src = src[sort_idx]
sorted_dst = dst[sort_idx]

# Count incoming edges per node
degree = torch.bincount(dst, minlength=data.num_nodes)

# Calculate starting position for each node
offsets = torch.zeros(data.num_nodes + 1, dtype=torch.long)
offsets[1:] = torch.cumsum(degree, dim=0)

print("Incoming-neighbour index created.")
print("Number of nodes:", data.num_nodes)
print("Total indexed edges:", len(sorted_src))
print("Index size:", sorted_src.numel())

Incoming-neighbour index created.
Number of nodes: 39357
Total indexed edges: 42445086
Index size: 42445086


In [12]:
import psutil

print(f"Available RAM: {psutil.virtual_memory().available / (1024**3):.2f} GB")

Available RAM: 1.91 GB


In [13]:
print("Edge index size:",
      data.edge_index.numel() * data.edge_index.element_size() / (1024**3),
      "GB")

print("Available RAM:",
      psutil.virtual_memory().available / (1024**3),
      "GB")

Edge index size: 0.6324810683727264 GB
Available RAM: 1.4132804870605469 GB


In [14]:
import gc
import psutil

gc.collect()

print(
    f"Available RAM after cleanup: "
    f"{psutil.virtual_memory().available / (1024**3):.2f} GB"
)

Available RAM after cleanup: 1.21 GB


DISK BACKED NEIGHBOUR INDEX 

In [15]:

from pathlib import Path
import numpy as np
import torch
import gc
import psutil

# Project directories
PROJECT_DIR = Path.cwd().parent
PROCESSED_DIR = PROJECT_DIR / "notebooks" / "data" / "processed"

# Separate folder for the neighbour index
INDEX_DIR = PROCESSED_DIR / "neighbour_index"
INDEX_DIR.mkdir(parents=True, exist_ok=True)

# Confirm the graph is loaded
num_nodes = data.num_nodes
num_edges = data.edge_index.size(1)

print("Nodes:", num_nodes)
print("Edges:", num_edges)
print("Index folder:", INDEX_DIR)
print("Folder exists:", INDEX_DIR.exists())

# Check available memory
print(f"Available RAM: {psutil.virtual_memory().available / (1024**3):.2f} GB")


Nodes: 39357
Edges: 42445086
Index folder: c:\Users\User\OneDrive\Desktop\Faud detection with limited resources\Fraude_detection_with_limited_resources\notebooks\data\processed\neighbour_index
Folder exists: True
Available RAM: 2.27 GB


In [16]:

import numpy as np
import torch
import gc
import math

# Configuration
CHUNK_SIZE = 250_000
NUM_BUCKETS = 256
BUCKET_SIZE = math.ceil(num_nodes / NUM_BUCKETS)

# Temporary and final file locations
TEMP_DIR = INDEX_DIR / "temporary_buckets"
TEMP_DIR.mkdir(parents=True, exist_ok=True)

OFFSETS_PATH = INDEX_DIR / "offsets.npy"
NEIGHBORS_PATH = INDEX_DIR / "neighbors.npy"

# Remove temporary files from an earlier incomplete run
for file in TEMP_DIR.glob("bucket_*.bin"):
    file.unlink()

# Count incoming edges for each node
counts = np.zeros(num_nodes, dtype=np.int64)

src_all = data.edge_index[0]
dst_all = data.edge_index[1]

print("Step 1: Partitioning edges into disk buckets...")

for start in range(0, num_edges, CHUNK_SIZE):
    end = min(start + CHUNK_SIZE, num_edges)

    src = src_all[start:end].numpy().astype(np.uint32, copy=False)
    dst = dst_all[start:end].numpy().astype(np.uint32, copy=False)

    # Count incoming neighbours
    counts += np.bincount(dst, minlength=num_nodes)

    # Assign edges to destination buckets
    bucket_ids = dst // BUCKET_SIZE

    for bucket in np.unique(bucket_ids):
        mask = bucket_ids == bucket

        # Store each edge as (destination, source)
        pairs = np.column_stack((dst[mask], src[mask])).astype(np.uint32)

        bucket_path = TEMP_DIR / f"bucket_{int(bucket):03d}.bin"

        with open(bucket_path, "ab") as f:
            pairs.tofile(f)

    if (start // CHUNK_SIZE + 1) % 20 == 0:
        print(f"Processed {end:,} / {num_edges:,} edges")

print("Step 1 complete.")

# Calculate the starting position of each node's neighbours
offsets = np.zeros(num_nodes + 1, dtype=np.int64)
offsets[1:] = np.cumsum(counts)

# Create disk-backed output array
neighbors = np.lib.format.open_memmap(
    NEIGHBORS_PATH,
    mode="w+",
    dtype=np.uint32,
    shape=(num_edges,)
)

print("Step 2: Sorting buckets and writing neighbour index...")

for bucket in range(NUM_BUCKETS):
    bucket_path = TEMP_DIR / f"bucket_{bucket:03d}.bin"

    if not bucket_path.exists():
        continue

    pairs = np.fromfile(bucket_path, dtype=np.uint32).reshape(-1, 2)

    # Sort by destination node
    pairs = pairs[np.argsort(pairs[:, 0], kind="stable")]

    first_node = bucket * BUCKET_SIZE
    last_node = min(first_node + BUCKET_SIZE, num_nodes)

    # All destinations in this bucket occupy one contiguous output range
    output_start = offsets[first_node]
    output_end = offsets[last_node]

    neighbors[output_start:output_end] = pairs[:, 1]

    del pairs
    bucket_path.unlink()

    if (bucket + 1) % 32 == 0:
        print(f"Processed bucket {bucket + 1}/{NUM_BUCKETS}")

neighbors.flush()

# Save offsets
np.save(OFFSETS_PATH, offsets)

# Clean up
del neighbors, counts, src_all, dst_all
gc.collect()

print("\nNeighbour index created!")
print("Offsets:", OFFSETS_PATH)
print("Neighbours:", NEIGHBORS_PATH)
print("Total indexed edges:", int(offsets[-1]))
print("Expected edges:", num_edges)


Step 1: Partitioning edges into disk buckets...
Processed 5,000,000 / 42,445,086 edges
Processed 10,000,000 / 42,445,086 edges
Processed 15,000,000 / 42,445,086 edges
Processed 20,000,000 / 42,445,086 edges
Processed 25,000,000 / 42,445,086 edges
Processed 30,000,000 / 42,445,086 edges
Processed 35,000,000 / 42,445,086 edges
Processed 40,000,000 / 42,445,086 edges
Step 1 complete.
Step 2: Sorting buckets and writing neighbour index...
Processed bucket 32/256
Processed bucket 64/256
Processed bucket 96/256
Processed bucket 128/256
Processed bucket 160/256
Processed bucket 192/256
Processed bucket 224/256
Processed bucket 256/256

Neighbour index created!
Offsets: c:\Users\User\OneDrive\Desktop\Faud detection with limited resources\Fraude_detection_with_limited_resources\notebooks\data\processed\neighbour_index\offsets.npy
Neighbours: c:\Users\User\OneDrive\Desktop\Faud detection with limited resources\Fraude_detection_with_limited_resources\notebooks\data\processed\neighbour_index\neigh

In [17]:

import numpy as np
from pathlib import Path

# Load the index using memory mapping
offsets = np.load(OFFSETS_PATH, mmap_mode="r")
neighbors = np.load(NEIGHBORS_PATH, mmap_mode="r")

# Function to retrieve incoming neighbours of a node
def get_incoming_neighbors(node_id):
    start = offsets[node_id]
    end = offsets[node_id + 1]

    return neighbors[start:end]

# Test a few nodes
test_nodes = [0, 1, 10, 100, 1000]

for node in test_nodes:
    node_neighbors = get_incoming_neighbors(node)

    print(
        f"Node {node}: "
        f"{len(node_neighbors)} incoming neighbours"
    )

# Verify total edge count
print("\nTotal indexed edges:", offsets[-1])
print("Index verification:", offsets[-1] == num_edges)

# Verify each retrieved neighbour points to the selected node
for node in test_nodes:
    node_neighbors = get_incoming_neighbors(node)

    if len(node_neighbors) > 0:
        sample = np.asarray(node_neighbors[:min(5, len(node_neighbors))])
        actual_destinations = data.edge_index[1, :].numpy()
        actual_sources = data.edge_index[0, :].numpy()

        valid = all(
            np.any((actual_sources == src) & (actual_destinations == node))
            for src in sample
        )

        print(f"Node {node} neighbour check:", valid)


Node 0: 31 incoming neighbours
Node 1: 847 incoming neighbours
Node 10: 244 incoming neighbours
Node 100: 64 incoming neighbours
Node 1000: 427 incoming neighbours

Total indexed edges: 42445086
Index verification: True
Node 0 neighbour check: True
Node 1 neighbour check: True
Node 10 neighbour check: True
Node 100 neighbour check: True
Node 1000 neighbour check: True


In [18]:

import numpy as np
import torch

def sample_neighbors(seed_nodes, num_neighbors=10):
    sampled_edges = []
    sampled_nodes = set(seed_nodes.tolist())

    for node in seed_nodes.tolist():
        start = offsets[node]
        end = offsets[node + 1]

        node_neighbors = neighbors[start:end]

        if len(node_neighbors) > num_neighbors:
            selected = np.random.choice(
                node_neighbors,
                size=num_neighbors,
                replace=False
            )
        else:
            selected = node_neighbors

        for neighbor in selected:
            neighbor = int(neighbor)
            sampled_edges.append((neighbor, node))
            sampled_nodes.add(neighbor)

    if sampled_edges:
        sampled_edge_index = torch.tensor(
            sampled_edges,
            dtype=torch.long
        ).t().contiguous()
    else:
        sampled_edge_index = torch.empty((2, 0), dtype=torch.long)

    sampled_nodes = torch.tensor(
        sorted(sampled_nodes),
        dtype=torch.long
    )

    return sampled_nodes, sampled_edge_index


# Reproducibility
np.random.seed(27)

# Select one mini-batch from training nodes
seed_batch = train_idx[:16]

sampled_nodes, sampled_edge_index = sample_neighbors(
    seed_batch,
    num_neighbors=10
)

print("Seed nodes:", len(seed_batch))
print("Sampled nodes:", len(sampled_nodes))
print("Sampled edges:", sampled_edge_index.size(1))
print("Edge index shape:", sampled_edge_index.shape)

print(
    "Maximum sampled neighbours per seed:",
    max(
        min(int(offsets[node + 1] - offsets[node]), 10)
        for node in seed_batch.tolist()
    )
)


Seed nodes: 16
Sampled nodes: 176
Sampled edges: 160
Edge index shape: torch.Size([2, 160])
Maximum sampled neighbours per seed: 10


In [19]:

np.random.seed(27)

for batch_number in range(3):
    start = batch_number * 16
    seed_batch = train_idx[start:start + 16]

    sampled_nodes, sampled_edges = sample_neighbors(
        seed_batch,
        num_neighbors=10
    )

    print(f"\nBatch {batch_number + 1}")
    print("Seed nodes:", len(seed_batch))
    print("Sampled nodes:", len(sampled_nodes))
    print("Sampled edges:", sampled_edges.size(1))

    # Verify fanout for every seed
    for node in seed_batch.tolist():
        start_idx = offsets[node]
        end_idx = offsets[node + 1]
        expected_max = min(end_idx - start_idx, 10)

        actual = ((sampled_edges[1] == node).sum()).item()

        assert actual == expected_max

print("\nAll three batches passed!")



Batch 1
Seed nodes: 16
Sampled nodes: 176
Sampled edges: 160

Batch 2
Seed nodes: 16
Sampled nodes: 176
Sampled edges: 160

Batch 3
Seed nodes: 16
Sampled nodes: 170
Sampled edges: 154

All three batches passed!


In [20]:

# Select 16 training nodes
seed_batch = train_idx[:16]

# Sample their incoming neighbours
sampled_nodes, sampled_edge_index = sample_neighbors(
    seed_batch,
    num_neighbors=10
)

# Extract features for sampled nodes
batch_x = data.x[sampled_nodes]

# Extract labels for seed nodes only
batch_y = data.y[seed_batch]

# Map global node IDs to local mini-batch IDs
global_to_local = {
    int(node): idx
    for idx, node in enumerate(sampled_nodes.tolist())
}

# Convert sampled edges to local node indices
local_edge_index = torch.tensor(
    [
        [global_to_local[int(node)] for node in sampled_edge_index[0]],
        [global_to_local[int(node)] for node in sampled_edge_index[1]]
    ],
    dtype=torch.long
)

# Find local positions of seed nodes
seed_local_idx = torch.tensor(
    [global_to_local[int(node)] for node in seed_batch],
    dtype=torch.long
)

print("Sampled feature shape:", batch_x.shape)
print("Local edge index shape:", local_edge_index.shape)
print("Seed local indices:", seed_local_idx.shape)
print("Seed labels shape:", batch_y.shape)

print("Feature values finite:", torch.isfinite(batch_x).all().item())
print("Local edge indices valid:",
      local_edge_index.max().item() < len(sampled_nodes))



Sampled feature shape: torch.Size([175, 10])
Local edge index shape: torch.Size([2, 160])
Seed local indices: torch.Size([16])
Seed labels shape: torch.Size([16])
Feature values finite: True
Local edge indices valid: True


In [22]:

import torch
import torch.nn as nn
import torch.nn.functional as F

class CategorySemanticDecomposition(nn.Module):
    def __init__(self, input_dim=10, hidden_dim=64):
        super().__init__()

        assert hidden_dim % 2 == 0

        half_dim = hidden_dim // 2

        self.input_projection = nn.Linear(input_dim, hidden_dim)
        self.normal_projection = nn.Linear(half_dim, half_dim)
        self.abnormal_projection = nn.Linear(half_dim, half_dim)

        self.activation = nn.LeakyReLU()

    def forward(self, x):
        h = self.activation(self.input_projection(x))

        first_half, second_half = torch.chunk(h, 2, dim=1)

        normal = self.normal_projection(first_half)
        abnormal = self.abnormal_projection(second_half)

        return normal, abnormal


In [23]:

# Initialize CSD
csd_batch = CategorySemanticDecomposition(
    input_dim=10,
    hidden_dim=64
).to(device)

# Move mini-batch features to device
batch_x = batch_x.to(device)

# Apply CSD
batch_normal, batch_abnormal = csd_batch(batch_x)

# Extract representations for the 16 seed nodes
seed_normal = batch_normal[seed_local_idx.to(device)]
seed_abnormal = batch_abnormal[seed_local_idx.to(device)]

print("Input features:", batch_x.shape)
print("Normal representations:", batch_normal.shape)
print("Abnormal representations:", batch_abnormal.shape)

print("Seed normal representations:", seed_normal.shape)
print("Seed abnormal representations:", seed_abnormal.shape)

print("Normal finite:", torch.isfinite(batch_normal).all().item())
print("Abnormal finite:", torch.isfinite(batch_abnormal).all().item())


Input features: torch.Size([175, 10])
Normal representations: torch.Size([175, 32])
Abnormal representations: torch.Size([175, 32])
Seed normal representations: torch.Size([16, 32])
Seed abnormal representations: torch.Size([16, 32])
Normal finite: True
Abnormal finite: True


In [24]:

from torch_geometric.utils import softmax

class DenoisingAttention(nn.Module):
    def __init__(self, hidden_dim=32, attention_dim=32):
        super().__init__()

        self.attention_projection = nn.Linear(
            hidden_dim, attention_dim
        )

        self.attention_score = nn.Linear(
            attention_dim, 1, bias=False
        )

    def forward(self, normal, abnormal, edge_index):
        src, dst = edge_index

        combined = normal[src] + abnormal[src]

        hidden = torch.tanh(
            self.attention_projection(combined)
        )

        scores = self.attention_score(hidden)

        alpha = softmax(
            scores,
            index=dst,
            num_nodes=normal.size(0)
        )

        messages = (
            alpha * normal[src]
            + (1 - alpha) * abnormal[src]
        )

        return messages, alpha


In [25]:

attention_batch = DenoisingAttention(
    hidden_dim=32,
    attention_dim=32
).to(device)

messages, alpha = attention_batch(
    batch_normal,
    batch_abnormal,
    local_edge_index.to(device)
)

print("Attention weights:", alpha.shape)
print("Denoised messages:", messages.shape)

print("Minimum attention:", alpha.min().item())
print("Maximum attention:", alpha.max().item())

print("Messages finite:", torch.isfinite(messages).all().item())
print("Attention finite:", torch.isfinite(alpha).all().item())


Attention weights: torch.Size([160, 1])
Denoised messages: torch.Size([160, 32])
Minimum attention: 0.04538426175713539
Maximum attention: 0.19592802226543427
Messages finite: True
Attention finite: True


In [26]:

class AggregationUpdate(nn.Module):
    def __init__(self, hidden_dim=32):
        super().__init__()
        self.update_projection = nn.Linear(
            hidden_dim, hidden_dim
        )

    def forward(self, center_features, messages, edge_index):
        num_nodes = center_features.size(0)

        src, dst = edge_index

        neighbour_sum = torch.zeros_like(center_features)

        neighbour_sum.index_add_(
            0, dst, messages
        )

        degree = torch.bincount(
            dst, minlength=num_nodes
        ).to(center_features.dtype)

        degree_norm = degree.clamp(min=1).sqrt().unsqueeze(1)

        combined = center_features + neighbour_sum / degree_norm

        updated_features = self.update_projection(combined)

        return updated_features


In [27]:

aggregation_batch = AggregationUpdate(
    hidden_dim=32
).to(device)

# Initial centre representation
center_features = batch_normal + batch_abnormal

# Aggregate and update
updated_features = aggregation_batch(
    center_features,
    messages,
    local_edge_index.to(device)
)

# Extract updated seed-node representations
seed_updated = updated_features[
    seed_local_idx.to(device)
]

print("Updated node representations:", updated_features.shape)
print("Updated seed representations:", seed_updated.shape)

print(
    "Updated features finite:",
    torch.isfinite(updated_features).all().item()
)


Updated node representations: torch.Size([175, 32])
Updated seed representations: torch.Size([16, 32])
Updated features finite: True


In [28]:

class CGNNClassifier(nn.Module):
    def __init__(self, input_dim=32, hidden_dim=64, num_classes=2):
        super().__init__()

        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.classifier(x)


In [29]:

classifier_batch = CGNNClassifier(
    input_dim=32,
    hidden_dim=64,
    num_classes=2
).to(device)

# Predict only the 16 seed nodes
seed_logits = classifier_batch(seed_updated)

# Convert logits into probabilities
seed_probabilities = F.softmax(seed_logits, dim=1)

# Predicted classes
seed_predictions = torch.argmax(seed_probabilities, dim=1)

print("Seed logits:", seed_logits.shape)
print("Seed probabilities:", seed_probabilities.shape)
print("Predictions:", seed_predictions.shape)

print("\nFirst 5 predictions:")
print(seed_predictions[:5])

print("\nFirst 5 probability distributions:")
print(seed_probabilities[:5])

print(
    "\nProbabilities sum to 1:",
    torch.allclose(
        seed_probabilities.sum(dim=1),
        torch.ones(16, device=device),
        atol=1e-6
    )
)


Seed logits: torch.Size([16, 2])
Seed probabilities: torch.Size([16, 2])
Predictions: torch.Size([16])

First 5 predictions:
tensor([0, 0, 0, 0, 0])

First 5 probability distributions:
tensor([[1., 0.],
        [1., 0.],
        [1., 0.],
        [1., 0.],
        [1., 0.]], grad_fn=<SliceBackward0>)

Probabilities sum to 1: True


In [30]:

# Actual labels of the 16 seed nodes
batch_y = batch_y.to(device)

# Calculate class weights using the full labelled training subset
labelled_labels = data.y[labelled_1]

class_counts = torch.bincount(
    labelled_labels,
    minlength=2
).float()

class_weights = (
    len(labelled_labels) / (2 * class_counts)
).to(device)

# Weighted cross-entropy loss
criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

# Calculate loss
ce_loss_batch = criterion(
    seed_logits,
    batch_y
)

print("Seed labels:", batch_y)
print("Class counts:", class_counts)
print("Class weights:", class_weights)
print("Cross-entropy loss:", ce_loss_batch.item())

print(
    "Loss finite:",
    torch.isfinite(ce_loss_batch).item()
)


Seed labels: tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
Class counts: tensor([21.,  1.])
Class weights: tensor([ 0.5238, 11.0000])
Cross-entropy loss: 0.0
Loss finite: True


In [31]:

print("Raw logits:")
print(seed_logits.detach())

print("\nProbabilities with higher precision:")
torch.set_printoptions(precision=10)
print(seed_probabilities.detach())

print("\nActual batch labels:")
print(batch_y)

print("\nPredicted classes:")
print(seed_predictions)

print("\nCross-entropy loss:")
print(ce_loss_batch.item())


Raw logits:
tensor([[ 147.6075, -306.9983],
        [  34.9750, -243.2997],
        [ -19.5853, -155.1248],
        [  23.3728, -143.6911],
        [  -6.3699, -260.7656],
        [  34.7849, -431.9873],
        [  11.3861, -245.3189],
        [   4.5831, -274.2254],
        [  38.7623, -121.0701],
        [ -19.8588,  -87.8055],
        [  25.8291, -110.0375],
        [  31.5014, -129.9174],
        [  -2.1776,  -70.0513],
        [ -22.5168, -154.2388],
        [  53.2017, -125.0912],
        [  54.8212,  -78.8758]])

Probabilities with higher precision:
tensor([[1.0000000000e+00, 0.0000000000e+00],
        [1.0000000000e+00, 0.0000000000e+00],
        [1.0000000000e+00, 0.0000000000e+00],
        [1.0000000000e+00, 0.0000000000e+00],
        [1.0000000000e+00, 0.0000000000e+00],
        [1.0000000000e+00, 0.0000000000e+00],
        [1.0000000000e+00, 0.0000000000e+00],
        [1.0000000000e+00, 0.0000000000e+00],
        [1.0000000000e+00, 0.0000000000e+00],
        [1.0000000000e+

In [32]:

print("CSD normal representation:")
print("Mean:", batch_normal.abs().mean().item())
print("Maximum:", batch_normal.abs().max().item())

print("\nCSD abnormal representation:")
print("Mean:", batch_abnormal.abs().mean().item())
print("Maximum:", batch_abnormal.abs().max().item())

print("\nAggregated representations:")
print("Mean:", updated_features.abs().mean().item())
print("Maximum:", updated_features.abs().max().item())

print("\nSeed representations:")
print("Mean:", seed_updated.abs().mean().item())
print("Maximum:", seed_updated.abs().max().item())

print("\nClassifier logits:")
print("Maximum absolute logit:", seed_logits.abs().max().item())


CSD normal representation:
Mean: 185.77783203125
Maximum: 13563.7822265625

CSD abnormal representation:
Mean: 319.2657470703125
Maximum: 21630.28125

Aggregated representations:
Mean: 366.8463439941406
Maximum: 16296.9072265625

Seed representations:
Mean: 822.6632080078125
Maximum: 3832.823486328125

Classifier logits:
Maximum absolute logit: 431.9872741699219


In [33]:

print("Original feature statistics")

print("Minimum:", data.x.min().item())
print("Maximum:", data.x.max().item())
print("Mean:", data.x.mean().item())
print("Standard deviation:", data.x.std().item())

print("\nStatistics for each feature:")

for i in range(data.x.shape[1]):
    feature = data.x[:, i]

    print(
        f"Feature {i}: "
        f"min={feature.min().item():.4f}, "
        f"max={feature.max().item():.4f}, "
        f"mean={feature.mean().item():.4f}, "
        f"std={feature.std().item():.4f}"
    )


Original feature statistics
Minimum: 0.0
Maximum: 508174.0
Mean: 712.4375610351562
Standard deviation: 4971.234375

Statistics for each feature:
Feature 0: min=11.0000, max=5000.0000, mean=673.4355, std=644.9841
Feature 1: min=1.0000, max=508174.0000, mean=4286.6191, std=13925.0098
Feature 2: min=0.0000, max=4990.0000, mean=51.2323, std=165.2820
Feature 3: min=0.0000, max=3108.0000, mean=28.8120, std=96.6853
Feature 4: min=0.0000, max=178817.0000, mean=1693.6097, std=5912.2300
Feature 5: min=1.0000, max=60229.0000, mean=186.5710, std=754.2632
Feature 6: min=0.0000, max=2350.0000, mean=45.3174, std=85.9960
Feature 7: min=0.0000, max=60229.0000, mean=158.1784, std=748.9879
Feature 8: min=0.0020, max=1.0000, mean=0.3493, std=0.2897
Feature 9: min=0.0012, max=0.9955, mean=0.2506, std=0.2614


In [34]:

# Calculate statistics using training nodes only
train_features = data.x[train_idx]

feature_mean = train_features.mean(dim=0)
feature_std = train_features.std(dim=0)

# Avoid division by zero
feature_std = feature_std.clamp(min=1e-8)

# Standardize all nodes using training statistics
x_standardized = (
    data.x - feature_mean
) / feature_std

# Check the result
print("Standardized feature statistics:")
print("Overall mean:", x_standardized.mean().item())
print("Overall standard deviation:", x_standardized.std().item())

print("\nFeature-wise statistics:")

for i in range(x_standardized.shape[1]):
    print(
        f"Feature {i}: "
        f"mean={x_standardized[:, i].mean().item():.4f}, "
        f"std={x_standardized[:, i].std().item():.4f}"
    )

print(
    "\nAll values finite:",
    torch.isfinite(x_standardized).all().item()
)

print(
    "Maximum absolute value:",
    x_standardized.abs().max().item()
)


Standardized feature statistics:
Overall mean: -0.007495319005101919
Overall standard deviation: 0.8920257687568665

Feature-wise statistics:
Feature 0: mean=-0.0030, std=0.9818
Feature 1: mean=-0.0066, std=0.8821
Feature 2: mean=-0.0040, std=0.9383
Feature 3: mean=-0.0136, std=0.8126
Feature 4: mean=-0.0184, std=0.8677
Feature 5: mean=-0.0152, std=0.6911
Feature 6: mean=-0.0173, std=0.9667
Feature 7: mean=-0.0142, std=0.6888
Feature 8: mean=0.0050, std=1.0004
Feature 9: mean=0.0124, std=1.0166

All values finite: True
Maximum absolute value: 55.22706985473633


In [35]:

# Extract standardized mini-batch features
batch_x_std = x_standardized[sampled_nodes].to(device)

# Apply CSD again
batch_normal_std, batch_abnormal_std = csd_batch(batch_x_std)

# Check representation magnitudes
print("Standardized input:")
print("Mean:", batch_x_std.abs().mean().item())
print("Maximum:", batch_x_std.abs().max().item())

print("\nNormal CSD representation:")
print("Mean:", batch_normal_std.abs().mean().item())
print("Maximum:", batch_normal_std.abs().max().item())

print("\nAbnormal CSD representation:")
print("Mean:", batch_abnormal_std.abs().mean().item())
print("Maximum:", batch_abnormal_std.abs().max().item())

print("\nNumerical checks:")
print("Normal finite:", torch.isfinite(batch_normal_std).all().item())
print("Abnormal finite:", torch.isfinite(batch_abnormal_std).all().item())


Standardized input:
Mean: 0.5012668967247009
Maximum: 16.684335708618164

Normal CSD representation:
Mean: 0.17311231791973114
Maximum: 3.1610898971557617

Abnormal CSD representation:
Mean: 0.16067466139793396
Maximum: 4.022355079650879

Numerical checks:
Normal finite: True
Abnormal finite: True


In [36]:

# Recalculate attention using standardized CSD outputs
messages_std, alpha_std = attention_batch(
    batch_normal_std,
    batch_abnormal_std,
    local_edge_index.to(device)
)

# Recalculate aggregation
center_features_std = batch_normal_std + batch_abnormal_std

updated_features_std = aggregation_batch(
    center_features_std,
    messages_std,
    local_edge_index.to(device)
)

# Extract seed representations
seed_updated_std = updated_features_std[
    seed_local_idx.to(device)
]

print("Attention messages:", messages_std.shape)
print("Attention weights:", alpha_std.shape)

print("\nUpdated representations:")
print("All nodes:", updated_features_std.shape)
print("Seed nodes:", seed_updated_std.shape)

print("\nMagnitude checks:")
print("Updated mean:", updated_features_std.abs().mean().item())
print("Updated maximum:", updated_features_std.abs().max().item())

print(
    "All finite:",
    torch.isfinite(updated_features_std).all().item()
)


Attention messages: torch.Size([160, 32])
Attention weights: torch.Size([160, 1])

Updated representations:
All nodes: torch.Size([175, 32])
Seed nodes: torch.Size([16, 32])

Magnitude checks:
Updated mean: 0.18025074899196625
Updated maximum: 3.993382692337036
All finite: True


In [37]:

# Predict using standardized representations
seed_logits_std = classifier_batch(seed_updated_std)

seed_probabilities_std = F.softmax(
    seed_logits_std, dim=1
)

seed_predictions_std = torch.argmax(
    seed_probabilities_std, dim=1
)

# Calculate weighted cross-entropy
ce_loss_std = criterion(
    seed_logits_std,
    batch_y
)

print("Logits shape:", seed_logits_std.shape)
print("Predictions:", seed_predictions_std)

print("\nFirst five probability distributions:")
print(seed_probabilities_std[:5].detach())

print("\nCross-entropy loss:", ce_loss_std.item())

print(
    "Maximum absolute logit:",
    seed_logits_std.abs().max().item()
)

print(
    "Loss finite:",
    torch.isfinite(ce_loss_std).item()
)


Logits shape: torch.Size([16, 2])
Predictions: tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

First five probability distributions:
tensor([[0.5417611599, 0.4582387805],
        [0.5429087281, 0.4570911825],
        [0.5359357595, 0.4640642405],
        [0.5372631550, 0.4627368152],
        [0.5346915126, 0.4653085470]])

Cross-entropy loss: 0.6189449429512024
Maximum absolute logit: 0.3170512020587921
Loss finite: True


In [38]:

# Collect all trainable model components
model_parameters = (
    list(csd_batch.parameters())
    + list(attention_batch.parameters())
    + list(aggregation_batch.parameters())
    + list(classifier_batch.parameters())
)

# Initialize Adam optimizer
optimizer = torch.optim.Adam(
    model_parameters,
    lr=0.01,
    weight_decay=1e-4
)

print("Optimizer:", type(optimizer).__name__)
print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Weight decay:", optimizer.param_groups[0]["weight_decay"])
print("Trainable parameters:", sum(
    p.numel() for p in model_parameters if p.requires_grad
))


Optimizer: Adam
Learning rate: 0.01
Weight decay: 0.0001
Trainable parameters: 7202


In [39]:

# Start training step
optimizer.zero_grad()

# Forward pass using standardized features
normal, abnormal = csd_batch(batch_x_std)

messages, alpha = attention_batch(
    normal,
    abnormal,
    local_edge_index.to(device)
)

center_features = normal + abnormal

updated = aggregation_batch(
    center_features,
    messages,
    local_edge_index.to(device)
)

seed_updated = updated[seed_local_idx.to(device)]

logits = classifier_batch(seed_updated)

# Supervised loss
loss = criterion(logits, batch_y)

# Backpropagation
loss.backward()

# Check gradients
components = {
    "CSD": csd_batch,
    "Attention": attention_batch,
    "Aggregation": aggregation_batch,
    "Classifier": classifier_batch
}

print("Loss:", loss.item())
print("\nGradient checks:")

for name, model in components.items():
    has_gradient = any(
        p.grad is not None for p in model.parameters()
    )

    finite_gradients = all(
        torch.isfinite(p.grad).all().item()
        for p in model.parameters()
        if p.grad is not None
    )

    print(
        f"{name}: gradients={has_gradient}, "
        f"finite={finite_gradients}"
    )

# Update parameters
optimizer.step()

print("\nOne optimizer update completed.")


Loss: 0.6189449429512024

Gradient checks:
CSD: gradients=True, finite=True
Attention: gradients=True, finite=True
Aggregation: gradients=True, finite=True
Classifier: gradients=True, finite=True

One optimizer update completed.


In [40]:

# Check that model parameters are finite after the update
for name, model in components.items():
    all_finite = all(
        torch.isfinite(p).all().item()
        for p in model.parameters()
    )

    print(f"{name} parameters finite: {all_finite}")

# Check total number of parameters
total_parameters = sum(
    p.numel() for p in model_parameters
)

print("\nTotal trainable parameters:", total_parameters)


CSD parameters finite: True
Attention parameters finite: True
Aggregation parameters finite: True
Classifier parameters finite: True

Total trainable parameters: 7202


In [41]:

# Extract the 109 labelled training nodes
contrastive_x = x_standardized[labelled_5].to(device)

contrastive_labels = data.y[labelled_5].to(device)

# Generate CSD representations
normal_contrastive, abnormal_contrastive = csd_batch(
    contrastive_x
)

print("Labelled features:", contrastive_x.shape)
print("Normal representations:", normal_contrastive.shape)
print("Abnormal representations:", abnormal_contrastive.shape)

print("\nLabel distribution:")
print(torch.bincount(contrastive_labels, minlength=2))

print(
    "\nRepresentations finite:",
    torch.isfinite(normal_contrastive).all().item()
    and torch.isfinite(abnormal_contrastive).all().item()
)


Labelled features: torch.Size([109, 10])
Normal representations: torch.Size([109, 32])
Abnormal representations: torch.Size([109, 32])

Label distribution:
tensor([104,   5])

Representations finite: True


In [42]:

def csd_contrastive_loss(
    normal_features,
    abnormal_features,
    labels,
    tau=0.8
):
    # Select the corresponding semantic subspace
    features = torch.where(
        (labels == 0).unsqueeze(1),
        normal_features,
        abnormal_features
    )

    # Normalize representations
    features = F.normalize(features, p=2, dim=1)

    # Pairwise cosine similarities
    similarity = torch.mm(
        features, features.T
    ) / tau

    # Identify positive and negative pairs
    same_class = labels.unsqueeze(0) == labels.unsqueeze(1)

    self_mask = torch.eye(
        len(labels),
        dtype=torch.bool,
        device=labels.device
    )

    positive_mask = same_class & ~self_mask
    negative_mask = ~same_class

    valid = (
        positive_mask.any(dim=1)
        & negative_mask.any(dim=1)
    )

    if not valid.any():
        return features.sum() * 0.0

    losses = []

    for k in torch.where(valid)[0]:
        positive_scores = similarity[k][positive_mask[k]]
        negative_scores = similarity[k][negative_mask[k]]

        denominator = torch.logsumexp(
            negative_scores, dim=0
        )

        pair_losses = -(positive_scores - denominator)
        losses.append(pair_losses.mean())

    return torch.stack(losses).mean()


# Calculate contrastive loss
csd_loss = csd_contrastive_loss(
    normal_contrastive,
    abnormal_contrastive,
    contrastive_labels,
    tau=0.8
)

print("CSD Contrastive Loss:", csd_loss.item())
print("Loss finite:", torch.isfinite(csd_loss).item())


CSD Contrastive Loss: 1.1295032501220703
Loss finite: True


In [43]:

# Clear previous gradients
csd_batch.zero_grad()

# Recalculate contrastive representations
normal_test, abnormal_test = csd_batch(contrastive_x)

# Calculate contrastive loss
csd_loss_test = csd_contrastive_loss(
    normal_test,
    abnormal_test,
    contrastive_labels,
    tau=0.8
)

# Backpropagation
csd_loss_test.backward()

# Inspect gradients
for name, parameter in csd_batch.named_parameters():
    if parameter.grad is not None:
        print(
            f"{name}: "
            f"gradient norm={parameter.grad.norm().item():.6f}, "
            f"finite={torch.isfinite(parameter.grad).all().item()}"
        )
    else:
        print(f"{name}: No gradient")

print("\nContrastive loss:", csd_loss_test.item())


input_projection.weight: gradient norm=0.837957, finite=True
input_projection.bias: gradient norm=0.680222, finite=True
normal_projection.weight: gradient norm=1.411339, finite=True
normal_projection.bias: gradient norm=1.110151, finite=True
abnormal_projection.weight: gradient norm=1.313228, finite=True
abnormal_projection.bias: gradient norm=1.156336, finite=True

Contrastive loss: 1.1295032501220703


In [44]:

class SubspaceFeaturePerturbation(nn.Module):
    def forward(self, normal, abnormal, alpha, edge_index):
        src, dst = edge_index

        # Extract neighbour representations
        normal_edges = normal[src]
        abnormal_edges = abnormal[src]

        # Randomly permute feature dimensions
        normal_indices = torch.rand(
            normal_edges.shape,
            device=normal.device
        ).argsort(dim=1)

        abnormal_indices = torch.rand(
            abnormal_edges.shape,
            device=abnormal.device
        ).argsort(dim=1)

        normal_fa = torch.gather(
            normal_edges, 1, normal_indices
        )

        abnormal_fa = torch.gather(
            abnormal_edges, 1, abnormal_indices
        )

        alpha = alpha.view(-1, 1)

        # Generate two perturbed views
        normal_perturbed = (
            alpha * normal_fa
            + (1 - alpha) * abnormal_edges
        )

        abnormal_perturbed = (
            alpha * normal_edges
            + (1 - alpha) * abnormal_fa
        )

        return normal_perturbed, abnormal_perturbed


In [45]:

# Initialize SFP
sfp_batch = SubspaceFeaturePerturbation().to(device)

# Generate two perturbed views
normal_perturbed, abnormal_perturbed = sfp_batch(
    batch_normal_std,
    batch_abnormal_std,
    alpha_std,
    local_edge_index.to(device)
)

print("Normal perturbed view:", normal_perturbed.shape)
print("Abnormal perturbed view:", abnormal_perturbed.shape)

print("\nNumerical checks:")
print(
    "Normal finite:",
    torch.isfinite(normal_perturbed).all().item()
)
print(
    "Abnormal finite:",
    torch.isfinite(abnormal_perturbed).all().item()
)

print("\nMagnitude checks:")
print("Normal mean:", normal_perturbed.abs().mean().item())
print("Abnormal mean:", abnormal_perturbed.abs().mean().item())


Normal perturbed view: torch.Size([160, 32])
Abnormal perturbed view: torch.Size([160, 32])

Numerical checks:
Normal finite: True
Abnormal finite: True

Magnitude checks:
Normal mean: 0.14613059163093567
Abnormal mean: 0.1460355818271637


In [46]:

# Aggregate the two perturbed views
normal_perturbed_nodes = aggregation_batch(
    center_features_std,
    normal_perturbed,
    local_edge_index.to(device)
)

abnormal_perturbed_nodes = aggregation_batch(
    center_features_std,
    abnormal_perturbed,
    local_edge_index.to(device)
)

# Extract seed-node representations
normal_perturbed_seeds = normal_perturbed_nodes[
    seed_local_idx.to(device)
]

abnormal_perturbed_seeds = abnormal_perturbed_nodes[
    seed_local_idx.to(device)
]

print("Normal perturbed nodes:", normal_perturbed_nodes.shape)
print("Abnormal perturbed nodes:", abnormal_perturbed_nodes.shape)

print("\nNormal perturbed seeds:", normal_perturbed_seeds.shape)
print("Abnormal perturbed seeds:", abnormal_perturbed_seeds.shape)

print(
    "\nAll finite:",
    torch.isfinite(normal_perturbed_nodes).all().item()
    and torch.isfinite(abnormal_perturbed_nodes).all().item()
)


Normal perturbed nodes: torch.Size([175, 32])
Abnormal perturbed nodes: torch.Size([175, 32])

Normal perturbed seeds: torch.Size([16, 32])
Abnormal perturbed seeds: torch.Size([16, 32])

All finite: True


In [47]:

def consistency_loss(
    original_logits,
    normal_logits,
    abnormal_logits,
    unlabelled_mask
):
    # Convert logits into classification probabilities
    original_prob = F.softmax(original_logits, dim=1)
    normal_prob = F.softmax(normal_logits, dim=1)
    abnormal_prob = F.softmax(abnormal_logits, dim=1)

    # Select only unlabelled centre nodes
    original = original_prob[unlabelled_mask]
    normal = normal_prob[unlabelled_mask]
    abnormal = abnormal_prob[unlabelled_mask]

    # Cosine similarities
    sim_normal = F.cosine_similarity(
        original, normal, dim=1
    )

    sim_abnormal = F.cosine_similarity(
        original, abnormal, dim=1
    )

    eps = 1e-8
    sim_abnormal = sim_abnormal.clamp(min=eps, max=1.0)

    loss = -(sim_normal * torch.log(sim_abnormal)).mean()

    return loss


In [48]:

# Identify which seed nodes are labelled
labelled_set = set(labelled_1.tolist())

unlabelled_mask = torch.tensor(
    [
        int(node) not in labelled_set
        for node in seed_batch.tolist()
    ],
    dtype=torch.bool
).to(device)

print("Total seed nodes:", len(seed_batch))
print("Labelled seed nodes:", (~unlabelled_mask).sum().item())
print("Unlabelled seed nodes:", unlabelled_mask.sum().item())


Total seed nodes: 16
Labelled seed nodes: 0
Unlabelled seed nodes: 16


In [49]:

# Recalculate original graph messages
messages_current, alpha_current = attention_batch(
    batch_normal_std,
    batch_abnormal_std,
    local_edge_index.to(device)
)

# Centre-node representations
center_features_std = batch_normal_std + batch_abnormal_std

# Original graph aggregation
original_nodes = aggregation_batch(
    center_features_std,
    messages_current,
    local_edge_index.to(device)
)

# Extract original seed representations
original_seeds = original_nodes[seed_local_idx.to(device)]

# Generate predictions for all three views
original_logits = classifier_batch(original_seeds)

normal_logits = classifier_batch(normal_perturbed_seeds)

abnormal_logits = classifier_batch(abnormal_perturbed_seeds)

# Calculate consistency loss
cl_loss = consistency_loss(
    original_logits,
    normal_logits,
    abnormal_logits,
    unlabelled_mask
)

print("Original logits:", original_logits.shape)
print("Normal-view logits:", normal_logits.shape)
print("Abnormal-view logits:", abnormal_logits.shape)

print("\nConsistency loss:", cl_loss.item())
print("Loss is finite:", torch.isfinite(cl_loss).item())


Original logits: torch.Size([16, 2])
Normal-view logits: torch.Size([16, 2])
Abnormal-view logits: torch.Size([16, 2])

Consistency loss: 0.011223229579627514
Loss is finite: True


In [50]:

def cgnn_total_loss(
    ce_loss,
    csd_loss,
    cl_loss,
    lambda_csd=1.0,
    beta_cl=1.0
):
    total_loss = (
        ce_loss
        + lambda_csd * csd_loss
        + beta_cl * cl_loss
    )

    return total_loss


In [52]:
# Use predictions from the original graph
ce_loss = criterion(original_logits, batch_y.to(device))

print("Cross-Entropy Loss:", ce_loss.item())

Cross-Entropy Loss: 0.38586971163749695


In [54]:

total_loss = cgnn_total_loss(
    ce_loss,
    csd_loss,
    cl_loss,
    lambda_csd=1.0,
    beta_cl=1.0
)

print("Cross-Entropy Loss:", ce_loss.item())
print("CSD Contrastive Loss:", csd_loss.item())
print("Consistency Loss:", cl_loss.item())
print("--------------------------------")
print("Total CGNN Loss:", total_loss.item())
print("Total loss is finite:", torch.isfinite(total_loss).item())


Cross-Entropy Loss: 0.38586971163749695
CSD Contrastive Loss: 1.1295032501220703
Consistency Loss: 0.011223229579627514
--------------------------------
Total CGNN Loss: 1.526596188545227
Total loss is finite: True


In [55]:

# ==========================================
# CGNN: 1-LABEL EXPERIMENT CONFIGURATION
# ==========================================

import numpy as np
import torch

# Select the 1-label experiment
labelled_nodes = labelled_1
unlabelled_nodes = unlabelled_1

# Create a lookup set for labelled node IDs
labelled_set = set(labelled_nodes.tolist())

# Verify the experiment configuration
print("EXPERIMENT: CGNN - 1 LABEL")
print("--------------------------------")

print("Total training nodes:", len(train_idx))
print("Labelled nodes:", len(labelled_nodes))
print("Unlabelled nodes:", len(unlabelled_nodes))

print("\nLabel distribution:")
print(torch.bincount(data.y[labelled_nodes], minlength=2))

# Verify no overlap
overlap = set(labelled_nodes.tolist()) & set(unlabelled_nodes.tolist())

print("\nLabelled/unlabelled overlap:", len(overlap))
print("Training nodes accounted for:",
      len(labelled_nodes) + len(unlabelled_nodes))

assert len(overlap) == 0
assert len(labelled_nodes) + len(unlabelled_nodes) == len(train_idx)

print("\nConfiguration verified successfully.")


EXPERIMENT: CGNN - 1 LABEL
--------------------------------
Total training nodes: 3935
Labelled nodes: 22
Unlabelled nodes: 3913

Label distribution:
tensor([21,  1])

Labelled/unlabelled overlap: 0
Training nodes accounted for: 3935

Configuration verified successfully.


In [56]:

# ==========================================
# CGNN: INTEGRATED BATCH PREPARATION
# ==========================================

def prepare_cgnn_batch(
    labelled_nodes,
    unlabelled_nodes,
    num_unlabelled=16,
    num_neighbors=10
):
    # 1. Select unlabelled centre nodes randomly
    random_indices = torch.randperm(len(unlabelled_nodes))[
        :num_unlabelled
    ]

    selected_unlabelled = unlabelled_nodes[random_indices]

    # 2. Combine labelled and unlabelled centre nodes
    seed_batch = torch.cat([
        labelled_nodes,
        selected_unlabelled
    ])

    # 3. Sample their neighbours
    sampled_nodes, sampled_edge_index = sample_neighbors(
        seed_batch,
        num_neighbors=num_neighbors
    )

    # 4. Extract standardized node features
    batch_x_std = x_standardized[sampled_nodes].to(device)

    # 5. Convert global node IDs to local IDs
    global_to_local = {
        int(node): idx
        for idx, node in enumerate(sampled_nodes.tolist())
    }

    local_edges = [
        [
            global_to_local[int(node)]
            for node in sampled_edge_index[0]
        ],
        [
            global_to_local[int(node)]
            for node in sampled_edge_index[1]
        ]
    ]

    local_edge_index = torch.tensor(
        local_edges,
        dtype=torch.long
    )

    # 6. Locate centre nodes in the sampled graph
    seed_local_idx = torch.tensor(
        [
            global_to_local[int(node)]
            for node in seed_batch.tolist()
        ],
        dtype=torch.long
    )

    # 7. Extract centre-node labels
    batch_y = data.y[seed_batch].to(device)

    # First 22 seeds are labelled; remaining 16 are unlabelled
    labelled_seed_idx = torch.arange(len(labelled_nodes))
    unlabelled_seed_idx = torch.arange(
        len(labelled_nodes),
        len(seed_batch)
    )

    return {
        "seed_batch": seed_batch,
        "sampled_nodes": sampled_nodes,
        "batch_x_std": batch_x_std,
        "local_edge_index": local_edge_index.to(device),
        "seed_local_idx": seed_local_idx.to(device),
        "batch_y": batch_y,
        "labelled_seed_idx": labelled_seed_idx.to(device),
        "unlabelled_seed_idx": unlabelled_seed_idx.to(device)
    }


# Test batch preparation
test_batch = prepare_cgnn_batch(
    labelled_1,
    unlabelled_1
)

print("Centre nodes:", len(test_batch["seed_batch"]))
print("Labelled centre nodes:", len(test_batch["labelled_seed_idx"]))
print("Unlabelled centre nodes:", len(test_batch["unlabelled_seed_idx"]))
print("Sampled graph nodes:", len(test_batch["sampled_nodes"]))
print("Local edges:", test_batch["local_edge_index"].shape)
print("Feature shape:", test_batch["batch_x_std"].shape)


Centre nodes: 38
Labelled centre nodes: 22
Unlabelled centre nodes: 16
Sampled graph nodes: 418
Local edges: torch.Size([2, 380])
Feature shape: torch.Size([418, 10])


In [57]:

# ==========================================
# CGNN: INTEGRATED FORWARD PASS
# ==========================================

def cgnn_forward(batch):
    x = batch["batch_x_std"]
    edge_index = batch["local_edge_index"]
    seed_idx = batch["seed_local_idx"]

    # 1. Category Semantic Decomposition
    normal, abnormal = csd_batch(x)

    # 2. Denoising Attention
    messages, alpha = attention_batch(
        normal,
        abnormal,
        edge_index
    )

    # 3. Original graph aggregation
    center_features = normal + abnormal

    original_nodes = aggregation_batch(
        center_features,
        messages,
        edge_index
    )

    # 4. Original centre-node predictions
    original_seeds = original_nodes[seed_idx]
    original_logits = classifier_batch(original_seeds)

    # 5. Subspace Feature Perturbation
    normal_perturbed, abnormal_perturbed = sfp_batch(
        normal,
        abnormal,
        alpha,
        edge_index
    )

    # 6. Aggregate perturbed views
    normal_nodes = aggregation_batch(
        center_features,
        normal_perturbed,
        edge_index
    )

    abnormal_nodes = aggregation_batch(
        center_features,
        abnormal_perturbed,
        edge_index
    )

    # 7. Extract centre-node representations
    normal_seeds = normal_nodes[seed_idx]
    abnormal_seeds = abnormal_nodes[seed_idx]

    # 8. Generate perturbed predictions
    normal_logits = classifier_batch(normal_seeds)
    abnormal_logits = classifier_batch(abnormal_seeds)

    return {
        "normal_features": normal,
        "abnormal_features": abnormal,
        "alpha": alpha,
        "original_logits": original_logits,
        "normal_logits": normal_logits,
        "abnormal_logits": abnormal_logits,
        "original_nodes": original_nodes
    }


# Test integrated forward pass
forward_output = cgnn_forward(test_batch)

for name, value in forward_output.items():
    print(name, value.shape)


normal_features torch.Size([418, 32])
abnormal_features torch.Size([418, 32])
alpha torch.Size([380, 1])
original_logits torch.Size([38, 2])
normal_logits torch.Size([38, 2])
abnormal_logits torch.Size([38, 2])
original_nodes torch.Size([418, 32])


In [58]:

# ==========================================
# CGNN: INTEGRATED LOSS CALCULATION
# ==========================================

def calculate_cgnn_losses(batch, output):
    
    # 1. Cross-Entropy Loss: labelled centre nodes only
    labelled_idx = batch["labelled_seed_idx"]

    labelled_logits = output["original_logits"][labelled_idx]
    labelled_targets = batch["batch_y"][labelled_idx]

    ce_loss = criterion(labelled_logits, labelled_targets)

    # 2. CSD Contrastive Loss: labelled nodes only
    labelled_global_ids = batch["seed_batch"][labelled_idx]

    labelled_features = x_standardized[labelled_global_ids].to(device)

    normal_labelled, abnormal_labelled = csd_batch(labelled_features)

    csd_loss = csd_contrastive_loss(
        normal_labelled,
        abnormal_labelled,
        labelled_targets,
        tau=0.8
    )

    # 3. Consistency Loss: unlabelled centre nodes only
    unlabelled_idx = batch["unlabelled_seed_idx"]

    cl_loss = consistency_loss(
        output["original_logits"],
        output["normal_logits"],
        output["abnormal_logits"],
        torch.isin(
            torch.arange(len(batch["seed_batch"]), device=device),
            unlabelled_idx
        )
    )

    # 4. Combined objective
    total_loss = cgnn_total_loss(
        ce_loss,
        csd_loss,
        cl_loss,
        lambda_csd=1.0,
        beta_cl=1.0
    )

    return ce_loss, csd_loss, cl_loss, total_loss


# Calculate integrated losses
ce_loss, csd_loss, cl_loss, total_loss = calculate_cgnn_losses(
    test_batch,
    forward_output
)

print("CE Loss:", ce_loss.item())
print("CSD Loss:", csd_loss.item())
print("Consistency Loss:", cl_loss.item())
print("--------------------------------")
print("Total CGNN Loss:", total_loss.item())

print("\nAll losses finite:",
      all(torch.isfinite(loss).item()
          for loss in [ce_loss, csd_loss, cl_loss, total_loss]))


CE Loss: 0.7577020525932312
CSD Loss: -0.683090090751648
Consistency Loss: 0.019502999261021614
--------------------------------
Total CGNN Loss: 0.09411495923995972

All losses finite: True


In [59]:

# ==========================================
# CGNN: FIRST INTEGRATED TRAINING UPDATE
# ==========================================

# Clear previous gradients
optimizer.zero_grad()

# Forward pass
forward_output = cgnn_forward(test_batch)

# Calculate all three losses
ce_loss, csd_loss, cl_loss, total_loss = calculate_cgnn_losses(
    test_batch,
    forward_output
)

# Backpropagation
total_loss.backward()

# Check gradients
components = {
    "CSD": csd_batch,
    "Attention": attention_batch,
    "Aggregation": aggregation_batch,
    "Classifier": classifier_batch
}

print("GRADIENT CHECK")
print("--------------------------------")

for name, model in components.items():
    gradients = [
        p.grad for p in model.parameters()
        if p.grad is not None
    ]

    finite = all(
        torch.isfinite(g).all().item()
        for g in gradients
    )

    print(name, "- gradients:", len(gradients),
          "| finite:", finite)

# Update parameters
optimizer.step()

# Final loss report
print("\nFIRST INTEGRATED TRAINING UPDATE")
print("--------------------------------")
print("CE Loss:", ce_loss.item())
print("CSD Loss:", csd_loss.item())
print("Consistency Loss:", cl_loss.item())
print("Total Loss:", total_loss.item())

print("\nOptimizer update completed.")


GRADIENT CHECK
--------------------------------
CSD - gradients: 6 | finite: True
Attention - gradients: 3 | finite: True
Aggregation - gradients: 2 | finite: True
Classifier - gradients: 4 | finite: True

FIRST INTEGRATED TRAINING UPDATE
--------------------------------
CE Loss: 0.7577020525932312
CSD Loss: -0.683090090751648
Consistency Loss: 0.020664909854531288
Total Loss: 0.09527686983346939

Optimizer update completed.


In [60]:

# ==========================================
# CGNN: 1-LABEL TRAINING LOOP
# ==========================================

import random

NUM_EPOCHS = 5
NUM_UNLABELLED = 16
NUM_NEIGHBORS = 10

epoch_history = []

for epoch in range(NUM_EPOCHS):

    csd_batch.train()
    attention_batch.train()
    aggregation_batch.train()
    classifier_batch.train()

    epoch_losses = []

    # Shuffle labelled nodes for each epoch
    shuffled_labelled = labelled_1[
        torch.randperm(len(labelled_1))
    ]

    # Shuffle unlabelled nodes
    shuffled_unlabelled = unlabelled_1[
        torch.randperm(len(unlabelled_1))
    ]

    # Divide unlabelled nodes into batches of 16
    num_batches = int(np.ceil(
        len(shuffled_unlabelled) / NUM_UNLABELLED
    ))

    for batch_number in range(num_batches):

        start = batch_number * NUM_UNLABELLED
        end = min(start + NUM_UNLABELLED,
                  len(shuffled_unlabelled))

        selected_unlabelled = shuffled_unlabelled[start:end]

        # Prepare graph batch with labelled and unlabelled centres
        batch = prepare_cgnn_batch(
            shuffled_labelled,
            selected_unlabelled,
            num_unlabelled=len(selected_unlabelled),
            num_neighbors=NUM_NEIGHBORS
        )

        optimizer.zero_grad()

        # Forward pass
        output = cgnn_forward(batch)

        # Calculate losses
        ce, csd, cl, total = calculate_cgnn_losses(
            batch, output
        )

        # Backpropagation
        total.backward()

        # Parameter update
        optimizer.step()

        epoch_losses.append([
            ce.item(),
            csd.item(),
            cl.item(),
            total.item()
        ])

    # Average losses
    avg_losses = np.mean(epoch_losses, axis=0)

    epoch_history.append({
        "epoch": epoch + 1,
        "CE": avg_losses[0],
        "CSD": avg_losses[1],
        "Consistency": avg_losses[2],
        "Total": avg_losses[3]
    })

    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
        f"CE: {avg_losses[0]:.4f} | "
        f"CSD: {avg_losses[1]:.4f} | "
        f"CL: {avg_losses[2]:.4f} | "
        f"Total: {avg_losses[3]:.4f}"
    )


Epoch 1/5 | CE: 0.0132 | CSD: -2.4723 | CL: 0.0132 | Total: -2.4459
Epoch 2/5 | CE: 0.9381 | CSD: -2.4165 | CL: 0.0563 | Total: -1.4220
Epoch 3/5 | CE: 0.2119 | CSD: -2.4323 | CL: 0.0692 | Total: -2.1512
Epoch 4/5 | CE: 0.6003 | CSD: -2.2428 | CL: 0.1073 | Total: -1.5352
Epoch 5/5 | CE: 0.7942 | CSD: -2.2904 | CL: 0.1926 | Total: -1.3035


In [61]:
print("Training completed.")
print("Number of epochs:", len(epoch_history))
print("Final epoch:", epoch_history[-1])

print("\nModel components:")
print("CSD:", sum(p.numel() for p in csd_batch.parameters()))
print("Attention:", sum(p.numel() for p in attention_batch.parameters()))
print("Aggregation:", sum(p.numel() for p in aggregation_batch.parameters()))
print("Classifier:", sum(p.numel() for p in classifier_batch.parameters()))

Training completed.
Number of epochs: 5
Final epoch: {'epoch': 5, 'CE': 0.7942117253128363, 'CSD': -2.290362611108897, 'Consistency': 0.19261006963322416, 'Total': -1.3035408156258719}

Model components:
CSD: 2816
Attention: 1088
Aggregation: 1056
Classifier: 2242


In [62]:
val_labels = data.y[val_idx]

print("Validation samples:", len(val_idx))
print("Normal samples:", (val_labels == 0).sum().item())
print("Fraud samples:", (val_labels == 1).sum().item())

Validation samples: 11807
Normal samples: 11266
Fraud samples: 541


In [63]:
import numpy as np
import torch

# Put all model components in evaluation mode
csd_batch.eval()
attention_batch.eval()
aggregation_batch.eval()
classifier_batch.eval()

# Store predictions for validation nodes
val_predictions = torch.empty(len(val_idx), dtype=torch.long)
val_true = data.y[val_idx].clone()

print("Validation evaluation prepared.")
print("Nodes to evaluate:", len(val_idx))
print("Batch size:", 16)

Validation evaluation prepared.
Nodes to evaluate: 11807
Batch size: 16


In [64]:
BATCH_SIZE = 16

val_nodes = val_idx.clone()
val_predictions = torch.empty(len(val_nodes), dtype=torch.long)

with torch.no_grad():
    for start in range(0, len(val_nodes), BATCH_SIZE):

        batch_nodes = val_nodes[start:start + BATCH_SIZE]

        # Sample neighbours for validation nodes
        sampled_nodes, sampled_edge_index = sample_neighbors(
            batch_nodes,
            num_neighbors=10
        )

        batch_x = x_standardized[sampled_nodes].to(device)

        # Convert global node IDs to local batch IDs
        global_to_local = {
            int(node): idx
            for idx, node in enumerate(sampled_nodes.tolist())
        }

        local_edges = [
            [global_to_local[int(node)] for node in sampled_edge_index[0]],
            [global_to_local[int(node)] for node in sampled_edge_index[1]]
        ]

        edge_index = torch.tensor(
            local_edges, dtype=torch.long
        ).to(device)

        seed_local_idx = torch.tensor(
            [global_to_local[int(node)] for node in batch_nodes.tolist()],
            dtype=torch.long
        ).to(device)

        # Forward pass
        normal, abnormal = csd_batch(batch_x)

        messages, alpha = attention_batch(
            normal, abnormal, edge_index
        )

        center_features = normal + abnormal

        node_embeddings = aggregation_batch(
            center_features, messages, edge_index
        )

        seed_embeddings = node_embeddings[seed_local_idx]

        logits = classifier_batch(seed_embeddings)

        predictions = logits.argmax(dim=1).cpu()

        val_predictions[start:start + len(batch_nodes)] = predictions

print("Validation predictions completed.")
print("Predictions:", len(val_predictions))
print("Predicted normal:", (val_predictions == 0).sum().item())
print("Predicted fraud:", (val_predictions == 1).sum().item())

Validation predictions completed.
Predictions: 11807
Predicted normal: 11167
Predicted fraud: 640


In [65]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    precision_score,
    confusion_matrix,
    classification_report
)

y_true = val_true.numpy()
y_pred = val_predictions.numpy()

print("VALIDATION RESULTS")
print("----------------------------")

print("Accuracy:", accuracy_score(y_true, y_pred))

print("Macro-F1:", f1_score(
    y_true, y_pred, average="macro", zero_division=0
))

print("Macro-Recall:", recall_score(
    y_true, y_pred, average="macro", zero_division=0
))

print("Macro-Precision:", precision_score(
    y_true, y_pred, average="macro", zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_true, y_pred,
    target_names=["Normal", "Fraud"],
    zero_division=0
))

VALIDATION RESULTS
----------------------------
Accuracy: 0.946726518167189
Macro-F1: 0.7196807292148115
Macro-Recall: 0.7389283772383036
Macro-Precision: 0.7037596825467897

Confusion Matrix:
[[10902   364]
 [  265   276]]

Classification Report:
              precision    recall  f1-score   support

      Normal       0.98      0.97      0.97     11266
       Fraud       0.43      0.51      0.47       541

    accuracy                           0.95     11807
   macro avg       0.70      0.74      0.72     11807
weighted avg       0.95      0.95      0.95     11807



In [66]:
BATCH_SIZE = 16

val_fraud_probabilities = torch.empty(len(val_idx))

csd_batch.eval()
attention_batch.eval()
aggregation_batch.eval()
classifier_batch.eval()

with torch.no_grad():
    for start in range(0, len(val_idx), BATCH_SIZE):

        batch_nodes = val_idx[start:start + BATCH_SIZE]

        sampled_nodes, sampled_edge_index = sample_neighbors(
            batch_nodes, num_neighbors=10
        )

        batch_x = x_standardized[sampled_nodes].to(device)

        global_to_local = {
            int(node): idx
            for idx, node in enumerate(sampled_nodes.tolist())
        }

        local_edges = [
            [global_to_local[int(node)] for node in sampled_edge_index[0]],
            [global_to_local[int(node)] for node in sampled_edge_index[1]]
        ]

        edge_index = torch.tensor(
            local_edges, dtype=torch.long
        ).to(device)

        seed_local_idx = torch.tensor(
            [global_to_local[int(node)] for node in batch_nodes.tolist()],
            dtype=torch.long
        ).to(device)

        normal, abnormal = csd_batch(batch_x)

        messages, alpha = attention_batch(
            normal, abnormal, edge_index
        )

        embeddings = aggregation_batch(
            normal + abnormal, messages, edge_index
        )

        logits = classifier_batch(embeddings[seed_local_idx])

        probabilities = torch.softmax(logits, dim=1)

        val_fraud_probabilities[start:start + len(batch_nodes)] = (
            probabilities[:, 1].cpu()
        )

print("Fraud probabilities collected:", len(val_fraud_probabilities))
print("Minimum probability:", val_fraud_probabilities.min().item())
print("Maximum probability:", val_fraud_probabilities.max().item())

Fraud probabilities collected: 11807
Minimum probability: 0.0
Maximum probability: 1.0


In [69]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score
)

y_true = val_true.numpy()
fraud_probs = val_fraud_probabilities.numpy()

threshold_results = []

for threshold in np.arange(0.10, 0.91, 0.05):

    predictions = (fraud_probs >= threshold).astype(int)

    threshold_results.append({
        "Threshold": round(threshold, 2),

        "Macro-F1": f1_score(
            y_true, predictions,
            average="macro",
            zero_division=0
        ),

        "Fraud Precision": precision_score(
            y_true, predictions,
            zero_division=0
        ),

        "Fraud Recall": recall_score(
            y_true, predictions,
            zero_division=0
        )
    })

threshold_df = pd.DataFrame(threshold_results)

print(threshold_df.round(4).to_string(index=False))

 Threshold  Macro-F1  Fraud Precision  Fraud Recall
      0.10    0.7321           0.4576        0.5287
      0.15    0.7321           0.4576        0.5287
      0.20    0.7321           0.4576        0.5287
      0.25    0.7321           0.4576        0.5287
      0.30    0.7321           0.4576        0.5287
      0.35    0.7321           0.4576        0.5287
      0.40    0.7321           0.4576        0.5287
      0.45    0.7321           0.4576        0.5287
      0.50    0.7321           0.4576        0.5287
      0.55    0.7321           0.4576        0.5287
      0.60    0.7321           0.4576        0.5287
      0.65    0.7321           0.4576        0.5287
      0.70    0.7321           0.4576        0.5287
      0.75    0.7321           0.4576        0.5287
      0.80    0.7321           0.4576        0.5287
      0.85    0.7321           0.4576        0.5287
      0.90    0.7321           0.4576        0.5287


In [70]:
print("Fraud probability statistics")
print("--------------------------------")

print("Minimum:", np.min(fraud_probs))
print("25th percentile:", np.percentile(fraud_probs, 25))
print("Median:", np.median(fraud_probs))
print("75th percentile:", np.percentile(fraud_probs, 75))
print("Maximum:", np.max(fraud_probs))

print("\nUnique probability values:",
      len(np.unique(fraud_probs)))

print("\nProbability ranges:")

for lower, upper in [(0, 0.1), (0.1, 0.3),
                     (0.3, 0.5), (0.5, 0.7),
                     (0.7, 0.9), (0.9, 1.0)]:

    count = np.sum(
        (fraud_probs >= lower) &
        (fraud_probs < upper if upper < 1.0
         else fraud_probs <= upper)
    )

    print(f"{lower:.1f} - {upper:.1f}: {count}")

Fraud probability statistics
--------------------------------
Minimum: 0.0
25th percentile: 0.0
Median: 0.0
75th percentile: 0.0
Maximum: 1.0

Unique probability values: 24

Probability ranges:
0.0 - 0.1: 11182
0.1 - 0.3: 0
0.3 - 0.5: 0
0.5 - 0.7: 0
0.7 - 0.9: 0
0.9 - 1.0: 625


In [71]:
# Inspect probabilities and logits
print("Probability diagnostics")
print("-----------------------")

print("Unique probabilities:")
print(np.unique(fraud_probs))

print("\nProbability counts:")
print(pd.Series(fraud_probs).value_counts().sort_index())

print("\nClassifier final layer:")
print(classifier_batch.classifier[-1])

print("\nClassifier weight statistics:")
final_weights = classifier_batch.classifier[-1].weight.detach().cpu()

print("Minimum:", final_weights.min().item())
print("Maximum:", final_weights.max().item())
print("Mean:", final_weights.mean().item())


Probability diagnostics
-----------------------
Unique probabilities:
[0.0000000e+00 3.0917644e-38 4.1435306e-35 6.5211045e-34 1.1141712e-33
 1.8931592e-33 1.3173107e-31 3.9554388e-30 1.2000814e-28 1.3279359e-24
 1.2709897e-22 1.4286165e-19 2.6481409e-15 3.9632473e-15 9.4652551e-14
 2.4885481e-11 5.0552379e-10 3.6567036e-07 3.3618785e-06 9.7782671e-01
 9.7957748e-01 9.9836713e-01 9.9999774e-01 1.0000000e+00]

Probability counts:
0.000000e+00    11164
3.091764e-38        1
4.143531e-35        1
6.521105e-34        1
1.114171e-33        1
1.893159e-33        1
1.317311e-31        1
3.955439e-30        1
1.200081e-28        1
1.327936e-24        1
1.270990e-22        1
1.428617e-19        1
2.648141e-15        1
3.963247e-15        1
9.465255e-14        1
2.488548e-11        1
5.055238e-10        1
3.656704e-07        1
3.361878e-06        1
9.778267e-01        1
9.795775e-01        1
9.983671e-01        1
9.999977e-01        1
1.000000e+00      621
Name: count, dtype: int64

Classifier f

In [72]:
# Inspect raw logits for validation batches
all_val_logits = []

csd_batch.eval()
attention_batch.eval()
aggregation_batch.eval()
classifier_batch.eval()

with torch.no_grad():
    for start in range(0, len(val_idx), 16):

        batch_nodes = val_idx[start:start + 16]

        sampled_nodes, sampled_edge_index = sample_neighbors(
            batch_nodes, num_neighbors=10
        )

        batch_x = x_standardized[sampled_nodes].to(device)

        global_to_local = {
            int(node): idx
            for idx, node in enumerate(sampled_nodes.tolist())
        }

        local_edges = [
            [global_to_local[int(node)] for node in sampled_edge_index[0]],
            [global_to_local[int(node)] for node in sampled_edge_index[1]]
        ]

        edge_index = torch.tensor(
            local_edges, dtype=torch.long
        ).to(device)

        seed_local_idx = torch.tensor(
            [global_to_local[int(node)] for node in batch_nodes.tolist()],
            dtype=torch.long
        ).to(device)

        normal, abnormal = csd_batch(batch_x)
        messages, alpha = attention_batch(
            normal, abnormal, edge_index
        )

        embeddings = aggregation_batch(
            normal + abnormal, messages, edge_index
        )

        logits = classifier_batch(embeddings[seed_local_idx])

        all_val_logits.append(logits.cpu())

all_val_logits = torch.cat(all_val_logits, dim=0)

print("Logit statistics")
print("----------------")
print("Minimum:", all_val_logits.min().item())
print("Maximum:", all_val_logits.max().item())
print("Mean:", all_val_logits.mean().item())
print("Standard deviation:", all_val_logits.std().item())

print("\nFirst 10 logits:")
print(all_val_logits[:10])

Logit statistics
----------------
Minimum: -63295.5078125
Maximum: 77462.1875
Mean: 757.9149780273438
Standard deviation: 9194.4052734375

First 10 logits:
tensor([[  7632.8647460938,  -6238.6582031250],
        [ 11434.0859375000,  -9342.2148437500],
        [  5588.3491210938,  -4565.6191406250],
        [  9232.3310546875,  -7545.5263671875],
        [ 11633.3417968750,  -9508.5712890625],
        [  4162.5205078125,  -3404.4699707031],
        [ 12768.6132812500, -10436.9951171875],
        [   854.3410644531,   -701.3013305664],
        [ 12199.1406250000,  -9975.2363281250],
        [  2443.9870605469,  -2005.4617919922]])


In [73]:
print("Embedding magnitude diagnostics")
print("--------------------------------")

print("Normal features:")
print("Mean absolute:", forward_output["normal_features"].abs().mean().item())
print("Maximum absolute:", forward_output["normal_features"].abs().max().item())

print("\nAbnormal features:")
print("Mean absolute:", forward_output["abnormal_features"].abs().mean().item())
print("Maximum absolute:", forward_output["abnormal_features"].abs().max().item())

print("\nAggregated embeddings:")
print("Mean absolute:", forward_output["original_nodes"].abs().mean().item())
print("Maximum absolute:", forward_output["original_nodes"].abs().max().item())

Embedding magnitude diagnostics
--------------------------------
Normal features:
Mean absolute: 0.17978909611701965
Maximum absolute: 3.2050154209136963

Abnormal features:
Mean absolute: 0.18034547567367554
Maximum absolute: 2.42983078956604

Aggregated embeddings:
Mean absolute: 0.2670178711414337
Maximum absolute: 3.4333558082580566


In [74]:
# Select 16 validation nodes
diagnostic_nodes = val_idx[:16]

sampled_nodes, sampled_edge_index = sample_neighbors(
    diagnostic_nodes,
    num_neighbors=10
)

batch_x = x_standardized[sampled_nodes].to(device)

global_to_local = {
    int(node): idx
    for idx, node in enumerate(sampled_nodes.tolist())
}

local_edges = [
    [global_to_local[int(node)] for node in sampled_edge_index[0]],
    [global_to_local[int(node)] for node in sampled_edge_index[1]]
]

edge_index = torch.tensor(local_edges, dtype=torch.long).to(device)

seed_local_idx = torch.tensor(
    [global_to_local[int(node)] for node in diagnostic_nodes.tolist()],
    dtype=torch.long
).to(device)

with torch.no_grad():
    normal, abnormal = csd_batch(batch_x)

    messages, alpha = attention_batch(
        normal, abnormal, edge_index
    )

    embeddings = aggregation_batch(
        normal + abnormal, messages, edge_index
    )

    seed_embeddings = embeddings[seed_local_idx]

    logits = classifier_batch(seed_embeddings)

print("Validation embedding diagnostics")
print("---------------------------------")
print("Input maximum:", batch_x.abs().max().item())
print("Normal maximum:", normal.abs().max().item())
print("Abnormal maximum:", abnormal.abs().max().item())
print("Message maximum:", messages.abs().max().item())
print("Aggregated maximum:", embeddings.abs().max().item())
print("Seed embedding maximum:", seed_embeddings.abs().max().item())
print("Logit maximum:", logits.abs().max().item())

Validation embedding diagnostics
---------------------------------
Input maximum: 11.041569709777832
Normal maximum: 22.765172958374023
Abnormal maximum: 310.0732116699219
Message maximum: 270.2926330566406
Aggregated maximum: 2710.1474609375
Seed embedding maximum: 1500.9537353515625
Logit maximum: 14547.470703125


In [75]:
print("CSD parameter diagnostics")
print("-------------------------")

for name, param in csd_batch.named_parameters():

    print(f"\n{name}")
    print("  Mean absolute:", param.detach().abs().mean().item())
    print("  Maximum absolute:", param.detach().abs().max().item())
    print("  All finite:", torch.isfinite(param).all().item())

CSD parameter diagnostics
-------------------------

input_projection.weight
  Mean absolute: 0.33341550827026367
  Maximum absolute: 2.0757901668548584
  All finite: True

input_projection.bias
  Mean absolute: 0.370058536529541
  Maximum absolute: 1.1046136617660522
  All finite: True

normal_projection.weight
  Mean absolute: 0.23851214349269867
  Maximum absolute: 1.6946449279785156
  All finite: True

normal_projection.bias
  Mean absolute: 0.25392112135887146
  Maximum absolute: 0.8239133358001709
  All finite: True

abnormal_projection.weight
  Mean absolute: 0.4843565821647644
  Maximum absolute: 1.9786250591278076
  All finite: True

abnormal_projection.bias
  Mean absolute: 0.36016786098480225
  Maximum absolute: 0.8501151204109192
  All finite: True


In [76]:
with torch.no_grad():

    diagnostic_x = batch_x

    # Stage 1: Input projection
    projected = csd_batch.input_projection(diagnostic_x)

    # Stage 2: Activation
    activated = csd_batch.activation(projected)

    # Stage 3: Split into two subspaces
    first_half, second_half = torch.chunk(
        activated, 2, dim=1
    )

    # Stage 4: Separate projections
    normal_output = csd_batch.normal_projection(first_half)
    abnormal_output = csd_batch.abnormal_projection(second_half)

    print("CSD stage diagnostics")
    print("---------------------")

    for name, tensor in [
        ("Input", diagnostic_x),
        ("Input projection", projected),
        ("Activated output", activated),
        ("First half", first_half),
        ("Second half", second_half),
        ("Normal output", normal_output),
        ("Abnormal output", abnormal_output)
    ]:

        print(
            f"{name:20s} | "
            f"Mean abs: {tensor.abs().mean().item():.4f} | "
            f"Max abs: {tensor.abs().max().item():.4f}"
        )

CSD stage diagnostics
---------------------
Input                | Mean abs: 0.5275 | Max abs: 11.0416
Input projection     | Mean abs: 1.1895 | Max abs: 41.2342
Activated output     | Mean abs: 0.7359 | Max abs: 41.2342
First half           | Mean abs: 0.4755 | Max abs: 15.5147
Second half          | Mean abs: 0.9962 | Max abs: 41.2342
Normal output        | Mean abs: 3.0786 | Max abs: 22.7652
Abnormal output      | Mean abs: 17.7419 | Max abs: 310.0732


In [77]:
with torch.no_grad():

    print("Second-half feature diagnostics")
    print("--------------------------------")

    for i in range(second_half.shape[1]):

        feature_values = second_half[:, i]

        print(
            f"Dimension {i:02d}: "
            f"Mean={feature_values.mean().item():.4f}, "
            f"Std={feature_values.std().item():.4f}, "
            f"Max={feature_values.abs().max().item():.4f}"
        )

Second-half feature diagnostics
--------------------------------
Dimension 00: Mean=0.7564, Std=1.4548, Max=12.4997
Dimension 01: Mean=1.1735, Std=1.8446, Max=10.4716
Dimension 02: Mean=0.0136, Std=0.1008, Max=0.7383
Dimension 03: Mean=1.5393, Std=2.4279, Max=18.9671
Dimension 04: Mean=0.8048, Std=1.2824, Max=5.2663
Dimension 05: Mean=0.7541, Std=0.9481, Max=2.7305
Dimension 06: Mean=1.8901, Std=4.3178, Max=37.5270
Dimension 07: Mean=1.3542, Std=2.0147, Max=12.4511
Dimension 08: Mean=0.0918, Std=0.4868, Max=5.8182
Dimension 09: Mean=0.7261, Std=1.2713, Max=5.7924
Dimension 10: Mean=1.5217, Std=2.5553, Max=19.7251
Dimension 11: Mean=0.5242, Std=0.8843, Max=5.2013
Dimension 12: Mean=1.0305, Std=1.2009, Max=3.5101
Dimension 13: Mean=2.6762, Std=5.0860, Max=38.3331
Dimension 14: Mean=0.7552, Std=1.0378, Max=4.5959
Dimension 15: Mean=0.6545, Std=1.2979, Max=8.4518
Dimension 16: Mean=3.0291, Std=5.5359, Max=41.2342
Dimension 17: Mean=0.2859, Std=0.5235, Max=2.5316
Dimension 18: Mean=0.7035, 